# Tecnológico de Monterrey
## Proyecto Integrador — Equipo 23
## Avance 5. Modelo final (ensambles)

**Integrantes:** 
- Edmundo Carmona Galindo (A01796647)
- Jaqueline Villafranca Hernández (A01656679)
- Edgar Omar Cruz Mendoza (A01795590)

---

### Problema

Cotejo entre fichas de búsqueda de personas desaparecidas (ante-mortem) y expedientes del registro forense
(post-mortem). Igual que en el Avance 4, es **vinculación de registros evaluada como ranking**: cada modelo puntúa
pares (ficha, expediente), se ordenan los candidatos de cada ficha y el perito revisa los primeros. El modelo
sugiere; el perito confirma con evidencia forense.

**Métrica principal: Recall@10** (fracción de fichas cuyo expediente correcto aparece entre los 10 primeros
candidatos). Un falso negativo es el error costoso: si el expediente correcto no está en la lista corta, el perito
no lo revisa.


| Sección | Contenido |
|---|---|
| 0 | Preparación de datos (idéntica al Avance 4) y contrato de datos |
| 5.1 | Modelos individuales de referencia (reproducción del Avance 4) |
| 5.2 | Protocolo de los ensambles |
| 5.3 | Ensambles homogéneos: Random Forest, Gradient Boosting, Bagging de regresión logística |
| 5.4 | Ensambles heterogéneos: votación suave, stacking y blending |
| 5.5 | Tabla comparativa |
| 5.6 | Prueba de estrés de los finalistas |
| 5.7 | Selección del modelo final |
| 5.8 | Gráficos del modelo final |
| 5.9 | Demostración para el perito |


In [ ]:
import pandas as pd
import numpy as np
import re
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import itertools
import scipy.stats as ss
import sklearn
warnings.filterwarnings('ignore')

from pathlib import Path
from scipy.stats.mstats import winsorize
from scipy.stats import skew
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_theme(style="whitegrid")
SEED = 42
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

# Las versiones se imprimen porque una de las celdas de 4.3 depende de ellas:
# MLPClassifier.fit solo acepta sample_weight desde scikit-learn 1.7.
print(f"Versiones -> pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")


## 0. Preparación de datos

Construye la base de expedientes, las fichas de consulta, el blocking, el vector de comparación de pares y
la selección de características. Produce el estado (`bl`, `Q_train`/`Q_test`, `P_train`/`P_test`, `FEATS_SEL`)
sobre el que se entrenan todos los modelos.

In [ ]:
# A. Carga del archivo
# (Resolución robusta de la ruta,
#  para que funcione tanto si el notebook corre desde Notebooks/ como desde la raíz del repo)
def _resolver_ruta_registro():
    nombre = "REGISTRO DE PERSONAS FALLECIDAS NO IDENTIFICADAS Y NO RECLAMADAS.xlsx"
    candidatos = [Path("..") / "Datos" / nombre, Path("Datos") / nombre]
    for c in candidatos:
        if c.exists():
            return c
    raise FileNotFoundError(f"No se encontró '{nombre}' en ../Datos ni en Datos/ (revisa el directorio de trabajo).")

file_path = _resolver_ruta_registro()
df_morge = pd.read_excel(file_path, header=0)

print(f"Dimensiones de la base de la Morgue: {df_morge.shape[0]} filas y {df_morge.shape[1]} columnas.")

# B. Fichas pre-mortem de DEMOSTRACIÓN (sintéticas; no corresponden a personas reales identificables).
# Se usan dos para que la demostración de 4.7 ejercite los dos extremos de una consulta real:
#   001 — ficha rica en datos, con señas individualizantes y municipio del propio estado, de modo que
#         la coincidencia de municipio pueda activarse (con un municipio de otro estado valdría 0 por
#         construcción y el ejemplo no mostraría esa característica funcionando).
#   002 — ficha pobre: sin señas particulares y con ropa genérica, que es como llegan muchas fichas reales.
ficha = pd.DataFrame([
    {
        'ID': 'FICHA-DEMO-001',
        'PRE_EDAD': 22.0,
        'PRE_ESTATURA': 170.0,
        'PRE_SEXO': 'HOMBRE',
        'PRE_FECHA': pd.to_datetime('2026-05-28'),
        'PRE_MUNICIPIO': 'QUERETARO',
        'PRE_SENAS': 'tatuaje en el antebrazo izquierdo cicatriz en la rodilla lunar en el cuello',
        'PRE_ROPA': 'playera gris shorts negro tenis blanco'
    },
    {
        'ID': 'FICHA-DEMO-002',
        'PRE_EDAD': 45.0,
        'PRE_ESTATURA': 160.0,
        'PRE_SEXO': 'MUJER',
        'PRE_FECHA': pd.to_datetime('2026-06-14'),
        'PRE_MUNICIPIO': 'SAN JUAN DEL RIO',
        'PRE_SENAS': '',
        'PRE_ROPA': 'blusa azul'
    },
])


In [ ]:
# En caso de que pandas haya metido una fila extra o basura antes de los nombres,
# la siguiente alternativa garantiza que tomamos la fila correcta (índice 0 de los datos) como título:
if 'AÑO' not in df_morge.columns:  # Verificación rápida
    df_morge.columns = df_morge.iloc[0]
    df_morge = df_morge.drop(index=0).reset_index(drop=True)

# Limpiamos los nombres de las columnas (quitando espacios en blanco al inicio o al final)
df_morge.columns = df_morge.columns.astype(str).str.strip()

# Por confidencialidad no se muestran filas reales: solo nombres de columnas
print(f"Dimensiones de la base de la Morgue: {df_morge.shape[0]} filas y {df_morge.shape[1]} columnas.")
display(pd.DataFrame({'Columna': df_morge.columns[:15]}))


In [ ]:
# =============================================================================
# 0.1 AUDITORÍA: filas vacías y fuga de información en la v1
# =============================================================================
import unicodedata
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.model_selection import learning_curve as sk_learning_curve
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import f_classif as sk_f_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
# Copia de trabajo sin columnas duplicadas (no altera df_morge de secciones anteriores)
df_a3 = df_morge.loc[:, ~pd.Index(df_morge.columns).duplicated()].copy()
PLACEHOLDERS = {'', 'SIN DATO', 'SIN DATOS', 'NINGUNA', 'NINGUNO', 'NO', 'N A', 'NA', 'NAN', 'NONE',
                'NO PRESENTA', 'NO SE OBSERVAN', 'NO SE OBSERVA', 'S D', 'SD', 'SIN', 'NO APLICA',
                'NO VALORABLE', 'POR VALORAR', 'DESCONOCIDO', 'X', '0'}

def norm_txt(x):
    # Mayúsculas, sin acentos, sin signos, espacios simples
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ''
    s = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^A-Z0-9 ]', ' ', s.upper())
    return re.sub(r'\s+', ' ', s).strip()

def limpiar_texto(x):
    s = norm_txt(x)
    return '' if s in PLACEHOLDERS else s

def buscar_col(df, exacta, *claves):
    # Devuelve la columna exacta o la primera cuyo nombre normalizado contiene todas las claves
    if exacta in df.columns:
        return exacta
    for c in df.columns:
        n = norm_txt(c)
        if all(k in n for k in claves):
            return c
    raise KeyError(f'No se encontró la columna {claves}')

COL = {
    'anio':        buscar_col(df_a3, 'AÑO', 'ANO'),
    'status':      buscar_col(df_a3, 'STATUS', 'STATUS'),
    'tipo':        buscar_col(df_a3, 'TIPO\n (QUÉ INGRESA A SEMEFO)', 'TIPO', 'INGRESA'),
    'sexo':        buscar_col(df_a3, 'SEXO', 'SEXO'),
    'edad':        buscar_col(df_a3, 'EDAD', 'EDAD'),
    'estatura':    buscar_col(df_a3, 'ESTATURA EN CM.', 'ESTATURA'),
    'municipio':   buscar_col(df_a3, 'MUNICIPIO DE PROCEDENCIA', 'MUNICIPIO', 'PROCEDENCIA'),
    'ropa':        buscar_col(df_a3, 'DESCRIPCION DE ROPA QUE VESTIA', 'ROPA', 'VESTIA'),
    'senas':       buscar_col(df_a3, 'SEÑAS PARTICULARES Y/O CARACTERISTICAS \n(CICATRICES, TATUAJES, LUNARES, MANCHAS, ETC.)', 'SENAS', 'PARTICULARES'),
    'nombre_ident': buscar_col(df_a3, 'EN CASO DE ESTAR IDENTIFICADO, NOMBRE QUE CORRESPONDE AL CADAVER\nDEBERA ESTABLECER AQUÍ EL NOMBRE EN EL MOMENTO EN QUE SE ACTUALICE DICHA SITUACION', 'CASO', 'IDENTIFICADO', 'NOMBRE'),
}
print('Columnas utilizadas en el baseline:')
for k, v in COL.items():
    print(f'  {k:<13} -> {v!r}'[:110])

# --- Diagnóstico
fila_valida = df_a3[COL['status']].notna() | df_a3[COL['anio']].notna()
campos_llenos = df_a3.notna().sum(axis=1)
y_v1 = df_a3[COL['nombre_ident']].map(limpiar_texto).ne('').astype(int)   # etiqueta usada en la v1

print(f"\nFilas totales del archivo: {len(df_a3):,}")
print(f"Filas con expediente (STATUS o AÑO no nulos): {fila_valida.sum():,}")
print(f"Filas vacías / de formato: {(~fila_valida).sum():,}  ({(~fila_valida).mean():.1%})")
print(f"Mediana de campos llenos -> expedientes: {campos_llenos[fila_valida].median():.0f} | filas vacías: {campos_llenos[~fila_valida].median():.0f}")

display(pd.crosstab(np.where(fila_valida, 'Expediente real', 'Fila vacía'),
                    np.where(y_v1 == 1, 'IDENTIFICADO (v1)', 'NO IDENTIFICADO (v1)'),
                    margins=True, margins_name='Total'))

auc_todo = roc_auc_score(y_v1, campos_llenos)
auc_valido = roc_auc_score(y_v1[fila_valida], campos_llenos[fila_valida])
print(f"AUC de un 'modelo' que solo cuenta campos llenos -> todas las filas: {auc_todo:.3f} | solo expedientes: {auc_valido:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(campos_llenos[fila_valida], color='steelblue', label='Expediente real', ax=ax[0], bins=40)
sns.histplot(campos_llenos[~fila_valida], color='salmon', label='Fila vacía', ax=ax[0], bins=40)
ax[0].set_title('Campos llenos por fila'); ax[0].set_xlabel('Número de columnas no nulas'); ax[0].legend()
ax[1].bar(['Todas las filas\n(v1)', 'Solo expedientes\n(v2)'], [auc_todo, auc_valido], color=['salmon', 'steelblue'], edgecolor='black')
ax[1].axhline(0.5, ls='--', color='gray', label='Azar')
for i, v in enumerate([auc_todo, auc_valido]):
    ax[1].text(i, v + 0.01, f'{v:.3f}', ha='center')
ax[1].set_ylim(0, 1.05); ax[1].set_title('AUC de "contar campos llenos" (señal de fuga)'); ax[1].legend()
plt.tight_layout(); plt.show()

print('\nCONCLUSIÓN AUTOMÁTICA:')
if auc_todo - auc_valido > 0.05:
    print(f"  Contar campos llenos predice la etiqueta v1 con AUC={auc_todo:.2f} en todas las filas, pero baja a {auc_valido:.2f} "
          "entre expedientes reales. Las métricas de la v1 están infladas por las filas vacías: la v1 se descarta como referencia.")
else:
    print("  No se detecta inflación relevante por filas vacías.")

In [ ]:
# =============================================================================
# 0.2 BASE LIMPIA (solo expedientes) + PSEUDO-FICHAS ANTE-MORTEM
# =============================================================================
from sklearn.model_selection import train_test_split

def parse_edad(x):
    if pd.isna(x): return np.nan
    s = str(x).upper().replace(',', '.')
    nums = [float(n) for n in re.findall(r'\d+(?:\.\d+)?', s)]
    if not nums: return np.nan
    if 'MES' in s:                      return nums[0] / 12
    if 'DIA' in s or 'SEMANA' in s:     return 0.0
    v = np.mean(nums[:2]) if len(nums) >= 2 and nums[1] < 120 else nums[0]   # rango "30-35" -> 32.5
    return v if 0 <= v <= 110 else np.nan

def parse_estatura(x):
    if pd.isna(x): return np.nan
    nums = re.findall(r'\d+(?:\.\d+)?', str(x).replace(',', '.'))
    if not nums: return np.nan
    v = float(nums[0])
    if v < 3: v *= 100                  # 1.65 m -> 165 cm
    return v if 40 <= v <= 220 else np.nan   # descarta capturas imposibles (p. ej. 7 cm)

def parse_sexo(x):
    s = norm_txt(x)
    if s.startswith(('MASC', 'HOMBRE')): return 'M'
    if s.startswith(('FEM', 'MUJER')):   return 'F'
    return 'ND'

exp = df_a3.loc[fila_valida]
bl = pd.DataFrame({
    'ID_INTERNO': exp.index.values,
    'SEXO':       exp[COL['sexo']].map(parse_sexo).values,
    'EDAD':       exp[COL['edad']].map(parse_edad).values,
    'ESTATURA':   exp[COL['estatura']].map(parse_estatura).values,
    'MUNICIPIO':  exp[COL['municipio']].map(limpiar_texto).values,
    'SENAS':      exp[COL['senas']].map(limpiar_texto).values,
    'ROPA':       exp[COL['ropa']].map(limpiar_texto).values,
    'TIPO':       exp[COL['tipo']].map(norm_txt).values,
})
status_n = exp[COL['status']].map(norm_txt)
nombre_ok = exp[COL['nombre_ident']].map(limpiar_texto).ne('')
# En la base el estatus positivo aparece como 'IDENTIFICADO NO RECLAMADO'; se toma cualquier estatus que inicie con IDENTIFICADO o con nombre registrado
bl['IDENTIFICADO'] = ((status_n.str.startswith('IDENTIFICADO')) | nombre_ok).astype(int).values

print(f"Expedientes en la base limpia: {len(bl):,}")
print(f"Identificados: {bl.IDENTIFICADO.sum():,} ({bl.IDENTIFICADO.mean():.1%}) | No identificados: {(1-bl.IDENTIFICADO).sum():,}")
print('\nConcordancia STATUS vs nombre registrado:')
display(pd.crosstab(status_n.values, nombre_ok.values, rownames=['STATUS'], colnames=['¿Tiene nombre?']))

cobertura = pd.DataFrame({
    c: bl.groupby('IDENTIFICADO')[c].apply(lambda s: (s.notna() & s.astype(str).ne('')).mean())
    for c in ['EDAD', 'ESTATURA', 'MUNICIPIO', 'SENAS', 'ROPA']
}).T.rename(columns={0: 'No identificados', 1: 'Identificados'})
print('\nCobertura de campos por clase (proporción con dato):')
display(cobertura.style.format('{:.1%}'))

# ---------------------------- Pseudo-fichas ----------------------------------
NIVELES_RUIDO = {
    'bajo':  dict(sd_edad=3, sd_est=3, p_falta_est=0.2, p_drop=0.3, p_typo=0.02, p_sin_ropa=0.2, p_mun=0.3, p_sexo=0.01),
    'medio': dict(sd_edad=5, sd_est=5, p_falta_est=0.4, p_drop=0.5, p_typo=0.05, p_sin_ropa=0.4, p_mun=0.5, p_sexo=0.02),
    'alto':  dict(sd_edad=8, sd_est=8, p_falta_est=0.6, p_drop=0.7, p_typo=0.10, p_sin_ropa=0.6, p_mun=0.7, p_sexo=0.05),
}
MUNICIPIOS = bl.loc[bl.MUNICIPIO.ne(''), 'MUNICIPIO'].unique()

def perturbar_texto(txt, p_drop, p_typo, rng):
    toks = txt.split()
    if not toks: return ''
    keep = [t for t in toks if rng.random() > p_drop] or [toks[rng.integers(len(toks))]]
    out = []
    for t in keep:
        if len(t) > 3 and rng.random() < p_typo:
            i = rng.integers(len(t)); t = t[:i] + chr(rng.integers(65, 91)) + t[i+1:]
        out.append(t)
    rng.shuffle(out)
    return ' '.join(out)

def generar_pseudofichas(src, nivel='medio', seed=SEED):
    p, rng = NIVELES_RUIDO[nivel], np.random.default_rng(seed)
    filas = []
    for _, r in src.iterrows():
        sexo = r.SEXO
        if sexo in ('M', 'F') and rng.random() < p['p_sexo']:
            sexo = 'F' if sexo == 'M' else 'M'
        edad = r.EDAD + rng.normal(0, p['sd_edad']) if pd.notna(r.EDAD) else np.nan
        est = np.nan if (pd.isna(r.ESTATURA) or rng.random() < p['p_falta_est']) else r.ESTATURA + rng.normal(0, p['sd_est'])
        mun = r.MUNICIPIO if (rng.random() >= p['p_mun'] or len(MUNICIPIOS) == 0) else rng.choice(MUNICIPIOS)
        filas.append(dict(QUERY_ID=int(r.ID_INTERNO), ID_VERDADERO=int(r.ID_INTERNO), SEXO=sexo, EDAD=edad, ESTATURA=est,
                          MUNICIPIO=mun, SENAS=perturbar_texto(r.SENAS, p['p_drop'], p['p_typo'], rng),
                          ROPA='' if rng.random() < p['p_sin_ropa'] else perturbar_texto(r.ROPA, p['p_drop'], p['p_typo'], rng)))
    return pd.DataFrame(filas)

ident = bl[bl.IDENTIFICADO == 1]
ids_train, ids_test = train_test_split(ident.ID_INTERNO.values, test_size=0.30, random_state=SEED)
Q_train = generar_pseudofichas(ident[ident.ID_INTERNO.isin(ids_train)], 'medio', seed=SEED)
Q_test  = generar_pseudofichas(ident[ident.ID_INTERNO.isin(ids_test)],  'medio', seed=SEED + 1)
print(f"\nPseudo-fichas -> entrenamiento: {len(Q_train)} | prueba: {len(Q_test)} | índice de búsqueda (libro completo): {len(bl):,} expedientes")
# El texto libre de señas y ropa proviene de expedientes reales y puede arrastrar nombres de terceros:
# se enmascara antes de mostrarlo (mismo criterio que ver_muestra() en el notebook post-mortem).
# El enmascarado conserva lo que ilustra el ejemplo —cuántas palabras sobreviven, de qué longitud y en
# qué orden tras la perturbación— sin revelar el contenido.
def enmascarar(txt, max_tokens=12):
    toks = str(txt).split()
    if not toks:
        return '(vacío)'
    vis = ' '.join(t[0] + '•' * (len(t) - 1) for t in toks[:max_tokens])
    return vis + (f' … (+{len(toks) - max_tokens} palabras)' if len(toks) > max_tokens else '')

def _redondear(v):
    return round(v, 1) if pd.notna(v) else v

print("Ejemplo de transformación (campos no identificantes; texto libre enmascarado):")
_ej = ident[ident.ID_INTERNO == Q_test.ID_VERDADERO.iloc[0]].iloc[0]
_q0 = Q_test.iloc[0]
display(pd.DataFrame(
    {'Expediente (post-mortem)': [_ej.SEXO, _redondear(_ej.EDAD), _redondear(_ej.ESTATURA),
                                  enmascarar(_ej.SENAS), enmascarar(_ej.ROPA)],
     'Pseudo-ficha (ante-mortem)': [_q0.SEXO, _redondear(_q0.EDAD), _redondear(_q0.ESTATURA),
                                    enmascarar(_q0.SENAS), enmascarar(_q0.ROPA)]},
    index=['Sexo', 'Edad', 'Estatura', 'Señas (enmascarada)', 'Ropa (enmascarada)']))

In [ ]:
# =============================================================================
# 0.3 FUNCIONES DE EVALUACIÓN (Recall@k, MRR, PR-AUC)
# =============================================================================
KS = (1, 5, 10, 20)

def rangos_verdaderos(pares, score_col, query_ids):
    # Rango (1 = primero) del expediente correcto para cada ficha. Empates: criterio pesimista.
    # Si el blocking eliminó al candidato correcto, rango = inf (cuenta como fallo).
    rangos = {}
    for qid, g in pares.groupby('QUERY_ID', sort=False):
        s = g[score_col].values; y = g['y'].values
        if y.sum() == 0:
            rangos[qid] = np.inf; continue
        st = s[y == 1][0]
        rangos[qid] = 1 + int(np.sum(s[y == 0] >= st))
    return np.array([rangos.get(q, np.inf) for q in query_ids], dtype=float)

def metricas_ranking(rangos, ks=KS):
    out = {f'Recall@{k}': float(np.mean(rangos <= k)) for k in ks}
    out['MRR'] = float(np.mean(np.where(np.isfinite(rangos), 1.0 / rangos, 0.0)))
    return out

def metricas_pares(y, score):
    return {'PR-AUC (pares)': average_precision_score(y, score), 'ROC-AUC (pares)': roc_auc_score(y, score)}

print("Funciones de evaluación definidas: Recall@k", KS, "+ MRR + PR-AUC/ROC-AUC a nivel par.")

In [ ]:
# =============================================================================
# 0.4 VECTORIZACIÓN DEL ÍNDICE Y CONSTRUCCIÓN DE PARES (con blocking)
# =============================================================================
tfidf_senas = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)
tfidf_ropa  = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)
XS_idx = tfidf_senas.fit_transform(bl.SENAS)
XR_idx = tfidf_ropa.fit_transform(bl.ROPA)

MARCAS = ['TATUA', 'CICATRI', 'LUNAR', 'PERFORA', 'PIERCING', 'MANCHA', 'AMPUTA', 'QUEMAD', 'VERRUG']
def matriz_marcas(textos):
    return np.array([[int(m in t) for m in MARCAS] for t in textos], dtype=float)
M_idx = matriz_marcas(bl.SENAS)

SEXO_idx, EDAD_idx, EST_idx = bl.SEXO.values, bl.EDAD.values.astype(float), bl.ESTATURA.values.astype(float)
MUN_idx, ID_idx, SENAS_VACIA_idx = bl.MUNICIPIO.values, bl.ID_INTERNO.values, bl.SENAS.eq('').values

FEATS = ['dif_edad', 'falta_edad', 'dif_estatura', 'falta_estatura', 'sexo_coincide',
         'municipio_coincide', 'sim_senas', 'sim_ropa', 'marcas_comunes', 'senas_vacia_exp', 'senas_vacia_ficha']
EDAD_MAX_BLOCK, IMPUT_DIF = 15, 20.0

def construir_pares(Q, usar_blocking=True):
    sim_s = (tfidf_senas.transform(Q.SENAS) @ XS_idx.T).toarray()
    sim_r = (tfidf_ropa.transform(Q.ROPA) @ XR_idx.T).toarray()
    marcas = matriz_marcas(Q.SENAS) @ M_idx.T
    bloques = []
    for i, q in enumerate(Q.itertuples(index=False)):
        d_edad = np.abs(EDAD_idx - q.EDAD) if pd.notna(q.EDAD) else np.full(len(bl), np.nan)
        d_est  = np.abs(EST_idx - q.ESTATURA) if pd.notna(q.ESTATURA) else np.full(len(bl), np.nan)
        mask = np.ones(len(bl), bool)
        if usar_blocking:
            if q.SEXO != 'ND':
                mask &= (SEXO_idx == q.SEXO) | (SEXO_idx == 'ND')
            mask &= np.isnan(d_edad) | (d_edad <= EDAD_MAX_BLOCK)
        idx = np.where(mask)[0]
        bloques.append(pd.DataFrame({
            'QUERY_ID': q.QUERY_ID, 'ID_CAND': ID_idx[idx],
            'y': (ID_idx[idx] == q.ID_VERDADERO).astype(int),
            'dif_edad': np.nan_to_num(d_edad[idx], nan=IMPUT_DIF), 'falta_edad': np.isnan(d_edad[idx]).astype(int),
            'dif_estatura': np.nan_to_num(d_est[idx], nan=IMPUT_DIF), 'falta_estatura': np.isnan(d_est[idx]).astype(int),
            'sexo_coincide': ((SEXO_idx[idx] == q.SEXO) & (q.SEXO != 'ND')).astype(int),
            'municipio_coincide': ((MUN_idx[idx] == q.MUNICIPIO) & (q.MUNICIPIO != '')).astype(int),
            'sim_senas': sim_s[i, idx], 'sim_ropa': sim_r[i, idx], 'marcas_comunes': marcas[i, idx],
            'senas_vacia_exp': SENAS_VACIA_idx[idx].astype(int),
            'senas_vacia_ficha': np.full(len(idx), int(q.SENAS == '')),
        }))
    return pd.concat(bloques, ignore_index=True)

P_train = construir_pares(Q_train)
P_test  = construir_pares(Q_test)

pc_train = P_train.groupby('QUERY_ID').y.max().reindex(Q_train.QUERY_ID).fillna(0).mean()
pc_test  = P_test.groupby('QUERY_ID').y.max().reindex(Q_test.QUERY_ID).fillna(0).mean()
rr = 1 - len(P_test) / (len(Q_test) * len(bl))
print(f"Pares de entrenamiento: {len(P_train):,} (positivos: {P_train.y.sum()}, prevalencia {P_train.y.mean():.3%})")
print(f"Pares de prueba:        {len(P_test):,} (positivos: {P_test.y.sum()}, prevalencia {P_test.y.mean():.3%})")
print(f"Blocking -> pair completeness train/test: {pc_train:.1%} / {pc_test:.1%} | reduction ratio: {rr:.1%}")
print(f"Candidatos promedio por ficha tras blocking: {len(P_test)/len(Q_test):.0f} de {len(bl):,}")
display(P_train.groupby('y')[FEATS].mean().T.rename(columns={0: 'Media pares NO coincidentes', 1: 'Media pares coincidentes'}).round(3))

In [ ]:
# =============================================================================
# 0.5 ENTRENAMIENTO Y EVALUACIÓN DE B0, B1, B2
# =============================================================================
def score_b1(P):
    cerc_edad = 1 - np.minimum(P.dif_edad, IMPUT_DIF) / IMPUT_DIF
    cerc_est  = 1 - np.minimum(P.dif_estatura, IMPUT_DIF) / IMPUT_DIF
    return 0.35*P.sim_senas + 0.25*P.sim_ropa + 0.20*cerc_edad*(1-P.falta_edad) + 0.10*cerc_est*(1-P.falta_estatura) + 0.10*P.municipio_coincide

B2_PARAMS = {'C': 1.0, 'class_weight': 'balanced'}
def nuevo_b2():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED, **B2_PARAMS))

# ---- Selección ligera de hiperparámetros por validación cruzada agrupada por ficha (solo datos de entrenamiento)
def cv_ranking(feats, params, n_splits=5):
    r10, mrr = [], []
    for tr_i, va_i in GroupKFold(n_splits=n_splits).split(P_train, P_train.y, groups=P_train.QUERY_ID):
        Ptr, Pva = P_train.iloc[tr_i], P_train.iloc[va_i].copy()
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED, **params)).fit(Ptr[feats], Ptr.y)
        Pva['s'] = m.predict_proba(Pva[feats])[:, 1]
        mr = metricas_ranking(rangos_verdaderos(Pva, 's', Pva.QUERY_ID.unique()))
        r10.append(mr['Recall@10']); mrr.append(mr['MRR'])
    return np.mean(r10), np.mean(mrr)
grid = [{'C': c, 'class_weight': cw} for c in (0.01, 0.1, 1.0) for cw in ('balanced', None)]
tabla_grid = pd.DataFrame([{**p, 'class_weight': str(p['class_weight']), **dict(zip(['Recall@10 (CV)', 'MRR (CV)'], cv_ranking(FEATS, p)))} for p in grid])
mejor = tabla_grid.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).iloc[0]
B2_PARAMS = {'C': float(mejor['C']), 'class_weight': None if mejor['class_weight'] == 'None' else 'balanced'}
print('Selección preliminar de hiperparámetros de la regresión logística (GroupKFold=5):')
display(tabla_grid.round(3))
print(f"Configuración elegida: {B2_PARAMS}")

rng_b0 = np.random.default_rng(SEED)
for P in (P_train, P_test):
    P['s_B0'] = rng_b0.random(len(P))
    P['s_B1'] = score_b1(P)

b2 = nuevo_b2().fit(P_train[FEATS], P_train.y)
P_train['s_B2'] = b2.predict_proba(P_train[FEATS])[:, 1]
P_test['s_B2']  = b2.predict_proba(P_test[FEATS])[:, 1]

filas, RANGOS_TEST = [], {}
for nombre, col in [('Azar (piso)', 's_B0'), ('Heurística TF-IDF (piso)', 's_B1'), ('Regresión logística', 's_B2')]:
    r_test = rangos_verdaderos(P_test, col, Q_test.QUERY_ID.values)
    RANGOS_TEST[nombre] = r_test
    filas.append({'Modelo': nombre, **metricas_ranking(r_test), **metricas_pares(P_test.y, P_test[col])})
res_baselines = pd.DataFrame(filas).set_index('Modelo')
rand_r10 = float(np.mean(np.minimum(10, P_test.groupby('QUERY_ID').size()) / P_test.groupby('QUERY_ID').size()))
print(f"Recall@10 teórico del azar (10 / candidatos por ficha): {rand_r10:.3f}")
print("\nPisos de referencia y regresión logística sobre el conjunto de prueba (ruido medio):")
display(res_baselines.round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ks_plot = np.arange(1, 51)
for nombre, r in RANGOS_TEST.items():
    ax[0].plot(ks_plot, [np.mean(r <= k) for k in ks_plot], lw=2, label=nombre)
ax[0].axvline(10, ls=':', color='gray'); ax[0].text(10.5, 0.03, 'k = 10 (carga del perito)', color='gray')
ax[0].set_xlabel('k (candidatos revisados por ficha)'); ax[0].set_ylabel('Recall@k'); ax[0].set_title('Curva Recall@k en prueba'); ax[0].legend(); ax[0].set_ylim(0, 1.02)
res_baselines[['Recall@1', 'Recall@10', 'MRR', 'PR-AUC (pares)']].T.plot(kind='bar', ax=ax[1], edgecolor='black', rot=0)
ax[1].set_title('Comparación de baselines (prueba)'); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# 0.6 IMPORTANCIA Y SELECCIÓN DE CARACTERÍSTICAS
# =============================================================================
q_fit, q_val = train_test_split(Q_train.QUERY_ID.values, test_size=0.25, random_state=SEED)
P_fit, P_val = P_train[P_train.QUERY_ID.isin(q_fit)], P_train[P_train.QUERY_ID.isin(q_val)]

def recall10_val(feats):
    m = nuevo_b2().fit(P_fit[feats], P_fit.y)
    Pv = P_val.assign(s=m.predict_proba(P_val[feats])[:, 1])
    return metricas_ranking(rangos_verdaderos(Pv, 's', q_val))['Recall@10'], average_precision_score(Pv.y, Pv.s), m

# ---- 1) FILTRO
rng_mi = np.random.default_rng(SEED)
neg_idx = P_fit.index[P_fit.y == 0]
sub = P_fit.loc[np.r_[P_fit.index[P_fit.y == 1], rng_mi.choice(neg_idx, min(30000, len(neg_idx)), replace=False)]]
mi = mutual_info_classif(sub[FEATS], sub.y, discrete_features=[f in ('falta_edad','falta_estatura','sexo_coincide','municipio_coincide','senas_vacia_exp','senas_vacia_ficha') for f in FEATS], random_state=SEED)
F_vals, p_vals = sk_f_classif(P_fit[FEATS], P_fit.y)

# ---- 2) EMBEBIDO: coeficientes estandarizados + trayectoria L1
r10_full, ap_full, m_full = recall10_val(FEATS)
coefs = pd.Series(m_full[-1].coef_[0], index=FEATS)
Cs = np.logspace(-4, 1, 12)
Xs_fit = StandardScaler().fit_transform(P_fit[FEATS])
trayectoria = pd.DataFrame([LogisticRegression(penalty='l1', solver='liblinear', C=c, class_weight='balanced', random_state=SEED)
                            .fit(Xs_fit, P_fit.y).coef_[0] for c in Cs], index=Cs, columns=FEATS)
primer_C_activo = {f: (trayectoria.index[trayectoria[f].abs() > 1e-6].min() if (trayectoria[f].abs() > 1e-6).any() else np.inf) for f in FEATS}

# ---- 3) ENVOLTURA: ablación + permutación
abl = {}
for f in FEATS:
    r10_sin, ap_sin, _ = recall10_val([x for x in FEATS if x != f])
    abl[f] = (r10_full - r10_sin, ap_full - ap_sin)
perm = permutation_importance(m_full, P_val[FEATS], P_val.y, scoring='average_precision', n_repeats=5, random_state=SEED, n_jobs=-1)

imp = pd.DataFrame({
    'Inf. mutua (filtro)': mi, 'F ANOVA (filtro)': F_vals,
    'Coef. estandarizado (embebido)': coefs.values, 'Odds ratio x1 DE': np.exp(coefs.values),
    'C de entrada L1 (menor = más útil)': [primer_C_activo[f] for f in FEATS],
    'Δ Recall@10 al quitarla (wrapper)': [abl[f][0] for f in FEATS], 'Δ PR-AUC al quitarla (wrapper)': [abl[f][1] for f in FEATS],
    'Permutación Δ PR-AUC': perm.importances_mean}, index=FEATS)
imp['Ranking promedio'] = pd.concat([imp['Inf. mutua (filtro)'].rank(ascending=False), imp['Coef. estandarizado (embebido)'].abs().rank(ascending=False),
                                     imp['Δ PR-AUC al quitarla (wrapper)'].rank(ascending=False), imp['Permutación Δ PR-AUC'].rank(ascending=False)], axis=1).mean(axis=1)
imp = imp.sort_values('Ranking promedio')
display(imp.round(4))

# ---- Visualización
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
imp['Inf. mutua (filtro)'].sort_values().plot(kind='barh', ax=axes[0, 0], color='slategray', edgecolor='black')
axes[0, 0].set_title('Filtro: información mutua con la etiqueta del par')
c_ord = imp['Coef. estandarizado (embebido)'].sort_values()
c_ord.plot(kind='barh', ax=axes[0, 1], color=['firebrick' if v < 0 else 'seagreen' for v in c_ord], edgecolor='black')
axes[0, 1].axvline(0, color='black', lw=0.8); axes[0, 1].set_title('Embebido: coeficientes estandarizados (verde ↑ coincidencia, rojo ↓)')
for f in FEATS:
    axes[1, 0].plot(trayectoria.index, trayectoria[f], marker='.', label=f)
axes[1, 0].set_xscale('log'); axes[1, 0].axhline(0, color='black', lw=0.8)
axes[1, 0].set_xlabel('C (inverso de la regularización L1)'); axes[1, 0].set_title('Embebido: trayectoria Lasso (L1)'); axes[1, 0].legend(fontsize=7, ncol=2)
imp[['Δ Recall@10 al quitarla (wrapper)', 'Δ PR-AUC al quitarla (wrapper)']].sort_values('Δ PR-AUC al quitarla (wrapper)').plot(kind='barh', ax=axes[1, 1], edgecolor='black')
axes[1, 1].axvline(0, color='black', lw=0.8); axes[1, 1].set_title('Envoltura: caída del desempeño al eliminar cada variable')
plt.tight_layout(); plt.show()

# ---- Decisión de selección (regla explícita y parsimoniosa)
descartables = [f for f in FEATS if abl[f][0] <= 0 and abl[f][1] <= 0.001 and perm.importances_mean[FEATS.index(f)] <= 0.001]
FEATS_SEL = [f for f in FEATS if f not in descartables] if descartables else FEATS.copy()
r10_sel, ap_sel, _ = recall10_val(FEATS_SEL)
if r10_sel < r10_full - 0.01:          # si la reducción empeora más de 1 pp, se conserva el conjunto completo
    FEATS_SEL, r10_sel, ap_sel = FEATS.copy(), r10_full, ap_full
print('\nCONCLUSIÓN AUTOMÁTICA (selección de características):')
print(f"  Top-3 por consenso de métodos: {list(imp.index[:3])}")
print(f"  Candidatas a eliminar (no aportan en ablación ni permutación): {descartables or 'ninguna'}")
print(f"  Conjunto final ({len(FEATS_SEL)} variables): {FEATS_SEL}")
print(f"  Validación -> completo: Recall@10={r10_full:.3f}, PR-AUC={ap_full:.3f} | seleccionado: Recall@10={r10_sel:.3f}, PR-AUC={ap_sel:.3f}")

In [ ]:
# =============================================================================
# 0.7 REGRESIÓN LOGÍSTICA sobre FEATS_SEL — es el octavo modelo de la tabla comparativa de 4.4. Se miden sus tiempos de entrenamiento e inferencia
# para que entre en la comparativa en igualdad de condiciones con los demás modelos.
# =============================================================================
t0 = time.perf_counter()
b2_final = nuevo_b2().fit(P_train[FEATS_SEL], P_train.y)
t_fit_logreg = time.perf_counter() - t0

t0 = time.perf_counter()
P_test['s_B2F'] = b2_final.predict_proba(P_test[FEATS_SEL])[:, 1]
t_inf_logreg = (time.perf_counter() - t0) / len(Q_test) * 1000
P_train['s_B2F'] = b2_final.predict_proba(P_train[FEATS_SEL])[:, 1]

_r10_lr = metricas_ranking(rangos_verdaderos(P_test, 's_B2F', Q_test.QUERY_ID.values))['Recall@10']
print(f"Regresión logística -> Recall@10 = {_r10_lr:.3f} | "
      f"entrenamiento = {t_fit_logreg:.3f} s | inferencia = {t_inf_logreg:.3f} ms/ficha")

## 4.1 Contrato de datos

Verificación de que la preparación produjo los objetos esperados. Si algún valor no coincide, la comparativa entre
modelos no es válida y la ejecución se detiene.

In [ ]:
# =============================================================================
# 4.1 CONTRATO DE DATOS — falla ruidosamente si la preparación no produjo los objetos esperados
# =============================================================================
errores = []

def _check(nombre, cond, detalle):
    if not cond:
        errores.append(f"  ✗ {nombre}: {detalle}")
    else:
        print(f"  ✓ {nombre}")

_check('len(df_a3) == 5317', len(df_a3) == 5317, f"obtenido {len(df_a3)}")
_check('fila_valida.sum() == 1247', fila_valida.sum() == 1247, f"obtenido {fila_valida.sum()}")
_check('bl.IDENTIFICADO.sum() == 355', bl.IDENTIFICADO.sum() == 355, f"obtenido {bl.IDENTIFICADO.sum()}")
_check('len(Q_train) == 248 and len(Q_test) == 107', len(Q_train) == 248 and len(Q_test) == 107,
       f"obtenido Q_train={len(Q_train)}, Q_test={len(Q_test)}")
_check('len(P_train) == 147_300 and len(P_test) == 64_008', len(P_train) == 147_300 and len(P_test) == 64_008,
       f"obtenido P_train={len(P_train):,}, P_test={len(P_test):,}")
_check('P_train.y.sum() == 238 and P_test.y.sum() == 105', P_train.y.sum() == 238 and P_test.y.sum() == 105,
       f"obtenido P_train.y.sum()={P_train.y.sum()}, P_test.y.sum()={P_test.y.sum()}")
_check('len(FEATS) == 11 and len(FEATS_SEL) == 10', len(FEATS) == 11 and len(FEATS_SEL) == 10,
       f"obtenido len(FEATS)={len(FEATS)}, len(FEATS_SEL)={len(FEATS_SEL)}")

if errores:
    print("\nCONTRATO DE DATOS VIOLADO:")
    print('\n'.join(errores))
    raise AssertionError(
        "El contrato de datos no se cumple. "
        "NO se debe ajustar el valor esperado: diagnosticar la causa (p. ej. una diferencia de "
        "comportamiento entre versiones de pandas/numpy) y corregirla para recuperar el comportamiento original."
    )
else:
    print("\nCONTRATO DE DATOS: OK — datos, pseudo-fichas, pares y características son los esperados.")
    print(f"Conjunto de características seleccionado (FEATS_SEL, {len(FEATS_SEL)} de {len(FEATS)}): {FEATS_SEL}")


## 5.1 Modelos individuales de referencia

Se reentrenan los ocho modelos individuales del Avance 4 con su configuración original, más las versiones
**ajustadas** del SVM lineal y del MLP. Reentrenarlos aquí, en la misma ejecución, garantiza que la tabla comparativa
de 5.5 mida tiempos y métricas en igualdad de condiciones con los ensambles.

**Modelos base del stacking y del blending.** Son los tres finalistas del Avance 4, con la configuración que se
eligió allí **por validación cruzada**:

| Modelo base | Configuración (Avance 4) | Recall@10 en prueba (Avance 4) |
|---|---|---|
| Regresión logística | `C = 0.01`, sin `class_weight` | 0.869 |
| SVM lineal (ajustado) | `C = 0.165`, `class_weight='balanced'`, `loss='hinge'` | 0.850 |
| MLP (ajustado) | 1 capa de 64, ReLU, `learning_rate_init = 8.4e-4`, `alpha = 7.6e-4`, lote 256, *early stopping* | 0.841 |

El SVM y el MLP **sin ajustar** rindieron algo mejor en prueba (0.860), pero elegirlos por eso sería seleccionar
con el conjunto de prueba. Se respeta la configuración que ganó en validación cruzada.

Los tres son complementarios: un modelo lineal probabilístico, uno de margen máximo y una red neuronal. Esa
diversidad de sesgos inductivos es la condición para que un ensamble heterogéneo aporte algo.

In [ ]:
# =============================================================================
# 5.1 (a) UTILIDADES COMUNES Y CONFIGURACIONES DEL AVANCE 4
# =============================================================================
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier, BaggingClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve, precision_recall_curve, brier_score_loss, confusion_matrix
from joblib import Parallel, delayed

# Configuraciones finales del Avance 4 (elegidas por GroupKFold, ver sección 4.5 de ese notebook)
MEJOR_SVM_A4 = {'C': 0.16484318387214908, 'class_weight': 'balanced', 'loss': 'hinge'}
MEJOR_MLP_A4 = {'hidden_layer_sizes': (64,), 'activation': 'relu', 'learning_rate_init': 0.0008436246579958732,
                'alpha': 0.0007554804756926672, 'batch_size': 256, 'early_stopping': True}

def build_svm(cfg):
    return make_pipeline(StandardScaler(), LinearSVC(C=cfg['C'], class_weight=cfg['class_weight'], loss=cfg['loss'],
                                                       max_iter=5000, random_state=SEED))

def build_mlp(cfg):
    return make_pipeline(StandardScaler(), MLPClassifier(
        hidden_layer_sizes=cfg['hidden_layer_sizes'], activation=cfg['activation'],
        learning_rate_init=cfg['learning_rate_init'], alpha=cfg['alpha'], batch_size=cfg['batch_size'],
        early_stopping=cfg['early_stopping'], max_iter=200, random_state=SEED))

# Constructores de los tres modelos base (finalistas del Avance 4)
BASES_A4 = {
    'Regresión logística':   lambda: nuevo_b2(),
    'SVM lineal (ajustado)': lambda: build_svm(MEJOR_SVM_A4),
    'MLP (ajustado)':        lambda: build_mlp(MEJOR_MLP_A4),
}

X_TR, Y_TR, QID_TR = P_train[FEATS_SEL], P_train.y.values, P_train.QUERY_ID.values
X_TE = P_test[FEATS_SEL]
N_SPLITS = 5
GKF_SPLITS = list(GroupKFold(n_splits=N_SPLITS).split(X_TR, Y_TR, groups=QID_TR))
TECHO_PROTOCOLO = float(pc_test)       # pair completeness del blocking en prueba
UMBRAL_NEGOCIO = 0.60                  # mismo umbral de negocio del Avance 4

def puntuar(modelo, X):
    """Score monótono por par: probabilidad si existe; si no, decision_function."""
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    return modelo.decision_function(X)

def rangos_rapidos(qid, y, s, query_ids):
    """Versión vectorizada de rangos_verdaderos (mismo criterio pesimista de empates). Se usa en la validación
    cruzada para acelerar la búsqueda; las métricas finales se calculan con rangos_verdaderos."""
    df = pd.DataFrame({'q': qid, 'y': y, 's': s})
    st = df.loc[df.y == 1].groupby('q').s.first().rename('st')
    df = df.join(st, on='q')
    cnt = df.loc[(df.y == 0) & (df.s >= df.st)].groupby('q').size()
    r = 1 + cnt.reindex(st.index, fill_value=0)
    return r.reindex(query_ids).astype(float).fillna(np.inf).values

REGISTRO = {}   # nombre -> resultados del modelo sobre el conjunto de prueba

def registrar(nombre, tipo, estrategia, modelo, t_fit, t_busqueda=np.nan):
    """Evalúa un modelo YA ENTRENADO sobre P_test completo y guarda métricas, scores, rangos y tiempos."""
    t0 = time.perf_counter()
    s_te = puntuar(modelo, X_TE)
    t_inf = (time.perf_counter() - t0) / len(Q_test) * 1000
    s_tr = puntuar(modelo, X_TR)
    r_te = rangos_verdaderos(P_test.assign(_s=s_te), '_s', Q_test.QUERY_ID.values)
    r_tr = rangos_verdaderos(P_train.assign(_s=s_tr), '_s', Q_train.QUERY_ID.values)
    met = metricas_ranking(r_te)
    met.update(metricas_pares(P_test.y, s_te))
    es_prob = hasattr(modelo, 'predict_proba')
    REGISTRO[nombre] = dict(
        tipo=tipo, estrategia=estrategia, modelo=modelo, s_test=s_te, r_test=r_te, es_prob=es_prob,
        fila={'Modelo': nombre, 'Tipo': tipo, 'Estrategia': estrategia, **met,
              'Brier (pares)': brier_score_loss(P_test.y, s_te) if es_prob else np.nan,
              'Tiempo entrenamiento (s)': t_fit, 'Tiempo búsqueda hiperparámetros (s)': t_busqueda,
              'Inferencia por ficha (ms)': t_inf,
              'Brecha Recall@10 (train−test)': metricas_ranking(r_tr)['Recall@10'] - met['Recall@10']})
    return REGISTRO[nombre]['fila']

def entrenar_registrar(nombre, tipo, estrategia, modelo, fit_params=None, t_busqueda=np.nan):
    t0 = time.perf_counter()
    modelo.fit(X_TR, Y_TR, **(fit_params or {}))
    t_fit = time.perf_counter() - t0
    f = registrar(nombre, tipo, estrategia, modelo, t_fit, t_busqueda)
    print(f"{nombre:<38} Recall@10={f['Recall@10']:.3f}  MRR={f['MRR']:.3f}  "
          f"entrenamiento={t_fit:.2f}s  inferencia={f['Inferencia por ficha (ms)']:.3f} ms/ficha")
    return modelo

# Verificación: la versión rápida de los rangos coincide con la del Avance 4
_r_a = rangos_verdaderos(P_test, 's_B1', Q_test.QUERY_ID.values)
_r_b = rangos_rapidos(P_test.QUERY_ID.values, P_test.y.values, P_test.s_B1.values, Q_test.QUERY_ID.values)
assert np.array_equal(_r_a, _r_b), "rangos_rapidos no coincide con rangos_verdaderos"
print(f"Utilidades listas. Techo del protocolo (pair completeness en prueba) = {TECHO_PROTOCOLO:.3f}. "
      f"rangos_rapidos verificado contra rangos_verdaderos.")

In [ ]:
# =============================================================================
# 5.1 (b) MODELOS INDIVIDUALES DEL AVANCE 4 (misma configuración) + pisos de referencia
# =============================================================================
IND = 'Individual (Avance 4)'
sw_bal = compute_sample_weight('balanced', Y_TR)

entrenar_registrar('Regresión logística', IND, 'Lineal probabilístico', nuevo_b2())
entrenar_registrar('Árbol de decisión (CART)', IND, 'Reglas',
                   make_pipeline(DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=50, random_state=SEED)))
entrenar_registrar('k-NN', IND, 'Instancias',
                   make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=50, weights='distance', n_jobs=-1)))
entrenar_registrar('Naive Bayes gaussiano', IND, 'Generativo', make_pipeline(StandardScaler(), GaussianNB()),
                   fit_params={'gaussiannb__sample_weight': sw_bal})
entrenar_registrar('SVM lineal', IND, 'Margen máximo',
                   make_pipeline(StandardScaler(), LinearSVC(class_weight='balanced', random_state=SEED, max_iter=5000)))
entrenar_registrar('SVM RBF escalable (Nystroem)', IND, 'Kernel',
                   make_pipeline(StandardScaler(), Nystroem(kernel='rbf', n_components=100, random_state=SEED),
                                 LinearSVC(class_weight='balanced', random_state=SEED, max_iter=5000)))
_mlp_a4 = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                                                        learning_rate_init=0.001, max_iter=300,
                                                        early_stopping=True, random_state=SEED))
try:
    entrenar_registrar('Perceptrón multicapa (MLP)', IND, 'Red neuronal', _mlp_a4,
                       fit_params={'mlpclassifier__sample_weight': sw_bal})
except TypeError:   # scikit-learn < 1.7 no acepta sample_weight en MLPClassifier
    entrenar_registrar('Perceptrón multicapa (MLP)', IND, 'Red neuronal', clone(_mlp_a4))
entrenar_registrar('LDA', IND, 'Discriminante lineal', make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()))
entrenar_registrar('SVM lineal (ajustado)', IND, 'Margen máximo', BASES_A4['SVM lineal (ajustado)']())
entrenar_registrar('MLP (ajustado)', IND, 'Red neuronal', BASES_A4['MLP (ajustado)']())

# Pisos de referencia (scores ya calculados en la sección 0.5)
PISOS = ['Azar (piso)', 'Heurística TF-IDF (piso)']
for nombre, col in [('Azar (piso)', 's_B0'), ('Heurística TF-IDF (piso)', 's_B1')]:
    r_te = rangos_verdaderos(P_test, col, Q_test.QUERY_ID.values)
    met = metricas_ranking(r_te); met.update(metricas_pares(P_test.y, P_test[col]))
    REGISTRO[nombre] = dict(tipo='Piso de referencia', estrategia='Sin entrenamiento', modelo=None,
                            s_test=P_test[col].values, r_test=r_te, es_prob=False,
                            fila={'Modelo': nombre, 'Tipo': 'Piso de referencia', 'Estrategia': 'Sin entrenamiento',
                                  **met, 'Brier (pares)': np.nan, 'Tiempo entrenamiento (s)': np.nan,
                                  'Tiempo búsqueda hiperparámetros (s)': np.nan, 'Inferencia por ficha (ms)': np.nan,
                                  'Brecha Recall@10 (train−test)': np.nan})

_ind = {k: v['fila'] for k, v in REGISTRO.items() if v['tipo'] == IND}
MEJOR_INDIVIDUAL = max(_ind, key=lambda k: (_ind[k]['Recall@10'], _ind[k]['MRR']))
print(f"\nMejor modelo individual (Recall@10, desempate por MRR): {MEJOR_INDIVIDUAL} "
      f"-> Recall@10 = {_ind[MEJOR_INDIVIDUAL]['Recall@10']:.3f}, MRR = {_ind[MEJOR_INDIVIDUAL]['MRR']:.3f}")
print("Es la referencia contra la que se mide cada ensamble en 5.5.")

## 5.2 Protocolo de los ensambles

Las cinco reglas del Avance 4 siguen vigentes (mismos pares, mismas 10 características, evaluación en `P_test`
completo, escalado dentro del *pipeline*, tiempos con `time.perf_counter()`). Se añaden tres para los ensambles:

1. **Optimización sin mirar la prueba.** Todos los hiperparámetros se eligen con `GroupKFold(5)` agrupado por ficha
   sobre `P_train`, con Recall@10 como criterio (desempate por MRR). Ninguna ficha aparece a la vez en
   entrenamiento y validación dentro de un pliegue.
2. **Predicciones fuera de pliegue (OOF) para los heterogéneos.** Los pesos de la votación y el meta-modelo del
   stacking se ajustan sobre las predicciones que cada modelo base hace de fichas que no vio. Si el meta-modelo
   aprendiera de predicciones dentro de la muestra, premiaría al modelo base que más se sobreajusta.
3. **Mejora "significativa" = diferencia pareada.** Cada ensamble se compara contra el mejor individual sobre las
   *mismas* 107 fichas con bootstrap pareado (B = 2,000). Una ganancia cuyo IC 95 % cruza cero no se declara mejora.

### Ensambles evaluados

| # | Ensamble | Estrategia | Tipo | Hiperparámetros optimizados |
|---|---|---|---|---|
| 1 | Random Forest | *Bagging* de árboles + submuestreo de variables | Homogéneo | árboles, profundidad, hoja mínima, `max_features`, `max_samples`, `class_weight` |
| 2 | Gradient Boosting (histogramas) | *Boosting* secuencial de árboles | Homogéneo | tasa de aprendizaje, iteraciones, hojas, hoja mínima, L2, `class_weight` |
| 3 | Bagging de regresión logística | *Bagging* del mejor individual | Homogéneo | estimadores, `max_samples`, `max_features`, *bootstrap* de filas y de columnas |
| 4 | Votación suave ponderada | Promedio ponderado de probabilidades | Heterogéneo | pesos de los 3 modelos base (rejilla sobre el símplex, paso 0.1) |
| 5 | Stacking | Meta-modelo sobre predicciones OOF | Heterogéneo | `C` y `class_weight` del meta-modelo, `passthrough` |
| 6 | Blending | Meta-modelo sobre un conjunto reservado | Heterogéneo | `C` y `class_weight` del meta-modelo, `passthrough` |

**Por qué estos seis.** El Random Forest y el Bagging reducen **varianza**: promedian modelos entrenados con
muestras distintas. El Gradient Boosting reduce **sesgo**: cada árbol corrige los errores del anterior y puede
capturar interacciones que un modelo lineal no ve (p. ej., que la similitud de ropa pese más cuando la ficha no trae
señas). Los heterogéneos combinan los tres finalistas del Avance 4: si cometen errores distintos en fichas
distintas, la combinación los compensa.

**El riesgo que motivó excluirlos en el Avance 4.** Con 0.16 % de pares positivos (238 en entrenamiento), un
ensamble de alta capacidad puede aprender ruido de las pseudo-fichas. Las tres reglas anteriores, junto con la
columna de brecha train−test y la prueba de estrés, permiten detectarlo.

In [ ]:
# =============================================================================
# 5.2 BÚSQUEDA ALEATORIA CON GroupKFold — scorer = Recall@10 (desempate MRR)
# =============================================================================
# Presupuesto de búsqueda (número de configuraciones por ensamble). Con ~8 núcleos la sección 5.3 completa toma
# del orden de 10-20 minutos; si el equipo tiene menos núcleos, se pueden reducir estos valores.
N_CONF_RF, N_CONF_HGB, N_CONF_BAG = 25, 30, 15

def _cv_config(build_fn, cfg, X, y, qid, splits):
    import warnings as _w
    _w.filterwarnings('ignore')
    r10s, mrrs = [], []
    for tr_i, va_i in splits:
        m = build_fn(cfg)
        m.fit(X.iloc[tr_i], y[tr_i])
        s = m.predict_proba(X.iloc[va_i])[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X.iloc[va_i])
        q_va = qid[va_i]
        r = rangos_rapidos(q_va, y[va_i], s, np.unique(q_va))
        r10s.append(np.mean(r <= 10)); mrrs.append(np.mean(np.where(np.isfinite(r), 1 / r, 0)))
    return float(np.mean(r10s)), float(np.mean(mrrs))

def busqueda_aleatoria_ens(nombre, muestrear_fn, build_fn, n_configs, n_jobs=-1):
    rng = np.random.default_rng(SEED)
    configs = muestrear_fn(n_configs, rng)
    t0 = time.perf_counter()
    res = Parallel(n_jobs=n_jobs)(delayed(_cv_config)(build_fn, cfg, X_TR, Y_TR, QID_TR, GKF_SPLITS) for cfg in configs)
    t_total = time.perf_counter() - t0
    tabla = pd.DataFrame([{**{k: (str(v) if isinstance(v, (tuple, type(None))) else v) for k, v in cfg.items()},
                           'Recall@10 (CV)': r10, 'MRR (CV)': mrr} for cfg, (r10, mrr) in zip(configs, res)])
    tabla['experimento'] = np.arange(1, len(tabla) + 1)
    tabla['mejor_hasta_ahora'] = tabla['Recall@10 (CV)'].cummax()
    idx = tabla.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).index[0]
    mejor = configs[idx]
    print(f"{nombre}: {n_configs} configuraciones x {N_SPLITS} pliegues = {n_configs * N_SPLITS} ajustes en {t_total:.1f} s")
    print(f"  Mejor configuración: {mejor}")
    print(f"  Recall@10 (CV) = {tabla.loc[idx, 'Recall@10 (CV)']:.3f} | MRR (CV) = {tabla.loc[idx, 'MRR (CV)']:.3f}")
    return tabla, mejor, t_total

# Referencia: el mejor individual evaluado con el MISMO protocolo de validación cruzada
R10_CV_REF, MRR_CV_REF = _cv_config(lambda cfg: BASES_A4['Regresión logística'](), None, X_TR, Y_TR, QID_TR, GKF_SPLITS)
print(f"Referencia (regresión logística del Avance 4) con este protocolo -> Recall@10 (CV) = {R10_CV_REF:.3f}, "
      f"MRR (CV) = {MRR_CV_REF:.3f}")
BUSQUEDAS = {}

## 5.3 Ensambles homogéneos

Tres ensambles del mismo tipo de estimador base. Los dos de árboles (Random Forest y Gradient Boosting) son
invariantes a la escala y no necesitan `StandardScaler`. Para que los tiempos de entrenamiento sean comparables con
los modelos individuales, el ajuste final se hace con un solo núcleo (`n_jobs=1`); la búsqueda de hiperparámetros sí
se paraleliza por configuración.

In [ ]:
# =============================================================================
# 5.3 (1) RANDOM FOREST — bagging de árboles con submuestreo de variables
# =============================================================================
def muestrear_rf(n, rng):
    return [{'n_estimators': int(rng.choice([100, 200, 300])),
             'max_depth': [None, 6, 10, 16][rng.integers(4)],
             'min_samples_leaf': int(rng.choice([1, 5, 20, 50, 100])),
             'max_features': ['sqrt', 0.5, 1.0][rng.integers(3)],
             'max_samples': [0.3, 0.6, None][rng.integers(3)],
             'class_weight': [None, 'balanced_subsample'][rng.integers(2)]} for _ in range(n)]

def build_rf(cfg, n_jobs=1):
    return RandomForestClassifier(**cfg, n_jobs=n_jobs, random_state=SEED)

tabla_rf, MEJOR_RF, t_bus_rf = busqueda_aleatoria_ens('Random Forest', muestrear_rf, build_rf, N_CONF_RF)
BUSQUEDAS['Random Forest'] = tabla_rf
rf_final = entrenar_registrar('Random Forest', 'Ensamble homogéneo', 'Bagging (árboles)', build_rf(MEJOR_RF),
                              t_busqueda=t_bus_rf)

In [ ]:
# =============================================================================
# 5.3 (2) GRADIENT BOOSTING (HistGradientBoosting) — boosting secuencial de árboles
# =============================================================================
def muestrear_hgb(n, rng):
    return [{'learning_rate': float(10 ** rng.uniform(-2.3, -0.5)),
             'max_iter': int(rng.choice([100, 200, 400])),
             'max_leaf_nodes': int(rng.choice([7, 15, 31, 63])),
             'min_samples_leaf': int(rng.choice([20, 50, 100, 200])),
             'l2_regularization': float(10 ** rng.uniform(-4, 1)),
             'class_weight': [None, 'balanced'][rng.integers(2)]} for _ in range(n)]

def build_hgb(cfg):
    # early_stopping=False: el número de iteraciones es un hiperparámetro explícito y el ajuste es reproducible
    return HistGradientBoostingClassifier(**cfg, early_stopping=False, random_state=SEED)

tabla_hgb, MEJOR_HGB, t_bus_hgb = busqueda_aleatoria_ens('Gradient Boosting', muestrear_hgb, build_hgb, N_CONF_HGB)
BUSQUEDAS['Gradient Boosting'] = tabla_hgb
hgb_final = entrenar_registrar('Gradient Boosting (HGB)', 'Ensamble homogéneo', 'Boosting (árboles)',
                               build_hgb(MEJOR_HGB), t_busqueda=t_bus_hgb)

In [ ]:
# =============================================================================
# 5.3 (3) BAGGING DE REGRESIÓN LOGÍSTICA — homogéneo sobre el mejor modelo individual
# =============================================================================
def muestrear_bag(n, rng):
    return [{'n_estimators': int(rng.choice([10, 25, 50])),
             'max_samples': float(rng.choice([0.3, 0.5, 0.8, 1.0])),
             'max_features': float(rng.choice([0.6, 0.8, 1.0])),
             'bootstrap': bool(rng.choice([True, False])),
             'bootstrap_features': bool(rng.choice([False, True]))} for _ in range(n)]

def build_bag(cfg):
    # Si no hay bootstrap de filas, max_samples=1.0 repetiría el mismo modelo: se fuerza submuestreo
    cfg = dict(cfg)
    if not cfg['bootstrap'] and cfg['max_samples'] >= 1.0:
        cfg['max_samples'] = 0.8
    return BaggingClassifier(estimator=nuevo_b2(), **cfg, n_jobs=1, random_state=SEED)

tabla_bag, MEJOR_BAG, t_bus_bag = busqueda_aleatoria_ens('Bagging de regresión logística', muestrear_bag, build_bag, N_CONF_BAG)
BUSQUEDAS['Bagging de regresión logística'] = tabla_bag
bag_final = entrenar_registrar('Bagging de regresión logística', 'Ensamble homogéneo', 'Bagging (lineal)',
                               build_bag(MEJOR_BAG), t_busqueda=t_bus_bag)

In [ ]:
# =============================================================================
# 5.3 (4) CONVERGENCIA DE LAS TRES BÚSQUEDAS
# =============================================================================
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (nombre, tabla) in zip(ax, BUSQUEDAS.items()):
    a.plot(tabla['experimento'], tabla['Recall@10 (CV)'], 'o', ms=4, alpha=0.45, color='gray', label='Configuración evaluada')
    a.plot(tabla['experimento'], tabla['mejor_hasta_ahora'], '-', lw=2.2, color='firebrick', label='Mejor hasta ahora')
    a.axhline(R10_CV_REF, ls='--', color='royalblue', lw=1.2, label=f'Regresión logística (CV) = {R10_CV_REF:.3f}')
    a.set_xlabel('Número de experimento'); a.set_ylabel('Recall@10 (CV, GroupKFold=5)')
    a.set_title(f'Convergencia — {nombre}'); a.legend(fontsize=7, loc='lower right')
    a.set_ylim(max(0, tabla['Recall@10 (CV)'].min() - 0.05), 1.0)
plt.tight_layout(); plt.show()

print('CONCLUSIÓN AUTOMÁTICA (5.3):')
for nombre, tabla in BUSQUEDAS.items():
    ultimo = int(tabla.loc[tabla['mejor_hasta_ahora'].diff().fillna(1) > 0, 'experimento'].iloc[-1])
    mejor = tabla['Recall@10 (CV)'].max()
    print(f"  {nombre}: mejor Recall@10 (CV) = {mejor:.3f} ({mejor - R10_CV_REF:+.3f} vs. regresión logística); "
          f"última mejora en el experimento {ultimo} de {len(tabla)} -> "
          f"{'la búsqueda se estabilizó.' if ultimo < len(tabla) else 'seguía mejorando: más configuraciones podrían ayudar.'}")
    print(f"     Dispersión entre configuraciones: {tabla['Recall@10 (CV)'].min():.3f}–{mejor:.3f} "
          f"({'el ensamble es sensible a sus hiperparámetros' if mejor - tabla['Recall@10 (CV)'].min() > 0.05 else 'el desempeño casi no depende de los hiperparámetros'}).")

## 5.4 Ensambles heterogéneos

Los tres combinan los **finalistas del Avance 4** (regresión logística, SVM lineal ajustado y MLP ajustado).

- **Votación suave ponderada.** La votación suave promedia probabilidades, y el SVM lineal no las produce: se envuelve
  en `CalibratedClassifierCV` (escalado de Platt, `method='sigmoid'`) para llevar su margen a la misma escala que las
  otras dos. Los pesos se eligen por rejilla sobre el símplex (paso 0.1, al menos dos modelos con peso > 0).
- **Stacking.** `StackingClassifier` con `cv` = los mismos pliegues `GroupKFold` agrupados por ficha. El meta-modelo
  es una regresión logística sobre los scores de los tres modelos base (probabilidad de la logística y del MLP,
  margen del SVM). Con `passthrough=True` además recibe las 10 características originales.
- **Blending.** Variante más simple del stacking: los modelos base se entrenan en el 75 % de las fichas de
  entrenamiento (`P_fit`) y el meta-modelo aprende de sus predicciones sobre el 25 % reservado (`P_val`), el mismo
  particionado por ficha de la sección 0.6. Usa menos datos que el stacking, pero no necesita reentrenar los modelos
  base en cada pliegue.

Primero se calculan, una sola vez, las predicciones fuera de pliegue de cada modelo base; con ellas se optimizan los
pesos de la votación y el meta-modelo del stacking sin volver a entrenar los modelos base en cada configuración.

In [ ]:
# =============================================================================
# 5.4 (0) PREDICCIONES FUERA DE PLIEGUE (OOF) DE LOS MODELOS BASE
# =============================================================================
def svm_calibrado():
    return CalibratedClassifierCV(BASES_A4['SVM lineal (ajustado)'](), method='sigmoid', cv=3)

def oof(build_fn, splits=GKF_SPLITS):
    out = np.full(len(Y_TR), np.nan)
    for tr_i, va_i in splits:
        m = build_fn().fit(X_TR.iloc[tr_i], Y_TR[tr_i])
        out[va_i] = puntuar(m, X_TR.iloc[va_i])
    return out

t0 = time.perf_counter()
OOF = pd.DataFrame({
    'lr_proba':  oof(BASES_A4['Regresión logística']),
    'svm_dec':   oof(BASES_A4['SVM lineal (ajustado)']),
    'svm_proba': oof(svm_calibrado),
    'mlp_proba': oof(BASES_A4['MLP (ajustado)']),
}, index=P_train.index)
t_oof = time.perf_counter() - t0

def r10_mrr_oof(s, qid=QID_TR, y=Y_TR):
    r = rangos_rapidos(qid, y, s, np.unique(qid))
    return float(np.mean(r <= 10)), float(np.mean(np.where(np.isfinite(r), 1 / r, 0)))

print(f"Predicciones OOF de los modelos base en {t_oof:.1f} s ({N_SPLITS} pliegues agrupados por ficha).")
print("Desempeño OOF de cada modelo base (fichas que el modelo no vio):")
for c in OOF.columns:
    r10, mrr = r10_mrr_oof(OOF[c].values)
    print(f"  {c:<10} Recall@10 = {r10:.3f} | MRR = {mrr:.3f}")

# Diversidad: correlación de rangos (Spearman) entre modelos base y desacuerdo en el top-10
corr_base = OOF[['lr_proba', 'svm_dec', 'mlp_proba']].corr(method='spearman')
hits = {c: (rangos_rapidos(QID_TR, Y_TR, OOF[c].values, np.unique(QID_TR)) <= 10) for c in ['lr_proba', 'svm_dec', 'mlp_proba']}
solo_uno = np.sum((hits['lr_proba'] + hits['svm_dec'] + hits['mlp_proba']) == 1)
alguno = np.mean(hits['lr_proba'] | hits['svm_dec'] | hits['mlp_proba'])
display(corr_base.round(3))
print(f"Fichas de entrenamiento que solo UNO de los tres modelos base coloca en su top-10: {solo_uno}")
print(f"Recall@10 'oráculo' (al menos un modelo base acierta): {alguno:.3f} -> es el techo que un ensamble "
      f"heterogéneo de estos tres podría alcanzar.")

In [ ]:
# =============================================================================
# 5.4 (1) VOTACIÓN SUAVE PONDERADA — pesos optimizados sobre OOF
# =============================================================================
P_OOF = OOF[['lr_proba', 'svm_proba', 'mlp_proba']].values
NOMBRES_VOTO = ['Regresión logística', 'SVM lineal (calibrado)', 'MLP (ajustado)']
t0 = time.perf_counter()
filas = []
for a in range(11):
    for b in range(11 - a):
        c = 10 - a - b
        w = np.array([a, b, c]) / 10
        if (w > 0).sum() < 2:
            continue                      # con un solo modelo no hay ensamble
        r10, mrr = r10_mrr_oof(P_OOF @ w)
        filas.append({'w_logistica': w[0], 'w_svm': w[1], 'w_mlp': w[2], 'Recall@10 (OOF)': r10, 'MRR (OOF)': mrr})
tabla_voto = pd.DataFrame(filas).sort_values(['Recall@10 (OOF)', 'MRR (OOF)'], ascending=False)
t_bus_voto = time.perf_counter() - t0 + t_oof
PESOS_VOTO = tabla_voto.iloc[0][['w_logistica', 'w_svm', 'w_mlp']].values.astype(float)
print(f"Rejilla de pesos: {len(tabla_voto)} combinaciones evaluadas. Las 5 mejores:")
display(tabla_voto.head(5).round(3))
print(f"Pesos elegidos -> logística {PESOS_VOTO[0]:.1f} | SVM {PESOS_VOTO[1]:.1f} | MLP {PESOS_VOTO[2]:.1f}")

_constructores_voto = [('lr', BASES_A4['Regresión logística']), ('svm', svm_calibrado), ('mlp', BASES_A4['MLP (ajustado)'])]
_activos = [(n, f()) for (n, f), w in zip(_constructores_voto, PESOS_VOTO) if w > 0]
voto_final = entrenar_registrar('Votación suave ponderada', 'Ensamble heterogéneo', 'Votación',
                                VotingClassifier(estimators=_activos, voting='soft',
                                                 weights=[w for w in PESOS_VOTO if w > 0]),
                                t_busqueda=t_bus_voto)

In [ ]:
# =============================================================================
# 5.4 (2) STACKING — meta-modelo optimizado sobre OOF; ajuste final con StackingClassifier(cv=GroupKFold)
# =============================================================================
M_OOF = OOF[['lr_proba', 'svm_dec', 'mlp_proba']].values       # mismo stack_method que usará StackingClassifier
GRID_META = [{'C': C, 'class_weight': cw, 'passthrough': pt}
             for C in (0.001, 0.01, 0.1, 1.0, 10.0, 100.0) for cw in (None, 'balanced') for pt in (False, True)]

def build_meta(cfg):
    return make_pipeline(StandardScaler(), LogisticRegression(C=cfg['C'], class_weight=cfg['class_weight'],
                                                              max_iter=3000, random_state=SEED))

def cv_meta(cfg, M, X, y, qid, splits):
    Z = np.column_stack([M, X.values]) if cfg['passthrough'] else M
    r10s, mrrs = [], []
    for tr_i, va_i in splits:
        m = build_meta(cfg).fit(Z[tr_i], y[tr_i])
        r = rangos_rapidos(qid[va_i], y[va_i], m.predict_proba(Z[va_i])[:, 1], np.unique(qid[va_i]))
        r10s.append(np.mean(r <= 10)); mrrs.append(np.mean(np.where(np.isfinite(r), 1 / r, 0)))
    return float(np.mean(r10s)), float(np.mean(mrrs))

t0 = time.perf_counter()
tabla_stack = pd.DataFrame([{**{k: str(v) for k, v in cfg.items()},
                             **dict(zip(['Recall@10 (CV)', 'MRR (CV)'], cv_meta(cfg, M_OOF, X_TR, Y_TR, QID_TR, GKF_SPLITS)))}
                            for cfg in GRID_META])
t_bus_stack = time.perf_counter() - t0 + t_oof
idx = tabla_stack.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).index[0]
MEJOR_META_STACK = GRID_META[idx]
print(f"Meta-modelo del stacking: {len(GRID_META)} configuraciones x {N_SPLITS} pliegues. Las 5 mejores:")
display(tabla_stack.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).head(5).round(3))
print(f"Configuración elegida: {MEJOR_META_STACK}")

stack_final = entrenar_registrar(
    'Stacking', 'Ensamble heterogéneo', 'Stacking (meta-logística)',
    StackingClassifier(estimators=[('lr', BASES_A4['Regresión logística']()),
                                   ('svm', BASES_A4['SVM lineal (ajustado)']()),
                                   ('mlp', BASES_A4['MLP (ajustado)']())],
                       final_estimator=build_meta(MEJOR_META_STACK), cv=GKF_SPLITS,
                       stack_method='auto', passthrough=MEJOR_META_STACK['passthrough']),
    t_busqueda=t_bus_stack)

coef_meta = stack_final.final_estimator_[-1].coef_[0]
nombres_meta = ['score logística', 'margen SVM', 'score MLP'] + (FEATS_SEL if MEJOR_META_STACK['passthrough'] else [])
print("Coeficientes estandarizados del meta-modelo (cuánto confía el stacking en cada fuente):")
display(pd.Series(coef_meta, index=nombres_meta).round(3).to_frame('Coeficiente'))

In [ ]:
# =============================================================================
# 5.4 (3) BLENDING — modelos base en P_fit (75 % de fichas), meta-modelo en P_val (25 %)
# =============================================================================
class Blending:
    """Blending con conjunto reservado agrupado por ficha. Expone predict_proba para usarse como cualquier modelo."""
    def __init__(self, bases, meta_cfg):
        self.bases, self.meta_cfg = bases, meta_cfg

    def _meta_X(self, X):
        M = np.column_stack([puntuar(m, X) for m in self.bases_.values()])
        return np.column_stack([M, X.values]) if self.meta_cfg['passthrough'] else M

    def fit(self, X_fit, y_fit, X_hold, y_hold):
        self.bases_ = {k: f().fit(X_fit, y_fit) for k, f in self.bases.items()}
        self.meta_ = build_meta(self.meta_cfg).fit(self._meta_X(X_hold), y_hold)
        return self

    def predict_proba(self, X):
        return self.meta_.predict_proba(self._meta_X(X))

X_FIT, Y_FIT = P_fit[FEATS_SEL], P_fit.y.values
X_VAL, Y_VAL, QID_VAL = P_val[FEATS_SEL], P_val.y.values, P_val.QUERY_ID.values
print(f"Blending -> fichas para modelos base: {P_fit.QUERY_ID.nunique()} | fichas reservadas para el meta-modelo: "
      f"{P_val.QUERY_ID.nunique()}")

# Optimización del meta-modelo: GroupKFold(5) DENTRO del conjunto reservado
t0 = time.perf_counter()
_bases_fit = {k: f().fit(X_FIT, Y_FIT) for k, f in BASES_A4.items()}
M_VAL = np.column_stack([puntuar(m, X_VAL) for m in _bases_fit.values()])
SPLITS_VAL = list(GroupKFold(n_splits=N_SPLITS).split(X_VAL, Y_VAL, groups=QID_VAL))
tabla_blend = pd.DataFrame([{**{k: str(v) for k, v in cfg.items()},
                             **dict(zip(['Recall@10 (CV)', 'MRR (CV)'], cv_meta(cfg, M_VAL, X_VAL, Y_VAL, QID_VAL, SPLITS_VAL)))}
                            for cfg in GRID_META])
t_bus_blend = time.perf_counter() - t0
idx = tabla_blend.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).index[0]
MEJOR_META_BLEND = GRID_META[idx]
print(f"Meta-modelo del blending: {len(GRID_META)} configuraciones. Las 5 mejores:")
display(tabla_blend.sort_values(['Recall@10 (CV)', 'MRR (CV)'], ascending=False).head(5).round(3))
print(f"Configuración elegida: {MEJOR_META_BLEND}")

t0 = time.perf_counter()
blend_final = Blending(BASES_A4, MEJOR_META_BLEND).fit(X_FIT, Y_FIT, X_VAL, Y_VAL)
t_fit_blend = time.perf_counter() - t0
f = registrar('Blending', 'Ensamble heterogéneo', 'Blending (meta-logística)', blend_final, t_fit_blend, t_bus_blend)
print(f"{'Blending':<38} Recall@10={f['Recall@10']:.3f}  MRR={f['MRR']:.3f}  entrenamiento={t_fit_blend:.2f}s  "
      f"inferencia={f['Inferencia por ficha (ms)']:.3f} ms/ficha")

## 5.5 Tabla comparativa

La tabla completa reúne los **10 modelos individuales** del Avance 4, los **6 ensambles** y los dos pisos de
referencia, ordenados por **Recall@10** (desempate por MRR). Métricas incluidas:

| Columna | Qué mide | Por qué importa en el proceso pericial |
|---|---|---|
| Recall@10 [IC 95 %] | Fichas con el expediente correcto entre los 10 primeros | Métrica principal: el falso negativo es irreversible |
| Recall@1 / @5 / @20 | Lo mismo con listas de 1, 5 y 20 | Sensibilidad al tamaño de la lista que revisa el perito |
| MRR | Inverso del rango promedio del correcto | Cuántos candidatos incorrectos se revisan antes del correcto |
| PR-AUC (pares) | Precisión-exhaustividad a nivel par | Con 0.16 % de positivos es más informativo que ROC-AUC |
| ROC-AUC (pares) | Separación global de pares | Complementaria; optimista con clases desbalanceadas |
| Brier (pares) | Error cuadrático de la probabilidad | Si el score se muestra como "confianza", debe estar calibrado |
| Tiempo de entrenamiento | Ajuste final sobre 147,300 pares | Costo de reentrenar cuando lleguen datos nuevos |
| Tiempo de búsqueda | Optimización de hiperparámetros | Costo de mantener el modelo |
| Inferencia por ficha | Puntuar ~600 candidatos de una ficha | Costo de cada consulta real |
| Brecha train−test | Recall@10 en entrenamiento menos en prueba | Señal de sobreajuste |

Después se presenta la **tabla que pide la rúbrica** (mejor individual + ensambles) y la diferencia pareada de cada
ensamble contra el mejor individual.

In [ ]:
# =============================================================================
# 5.5 (a) TABLA COMPARATIVA COMPLETA + IC BOOTSTRAP DE RECALL@10
# =============================================================================
B = 2000
rng_bs = np.random.default_rng(SEED)
BOOT_IDX = rng_bs.integers(0, len(Q_test), size=(B, len(Q_test)))

def bootstrap(r):
    hits = (r <= 10).astype(float)
    rr = np.where(np.isfinite(r), 1 / r, 0)
    return hits[BOOT_IDX].mean(1), rr[BOOT_IDX].mean(1)

BOOT = {k: bootstrap(v['r_test']) for k, v in REGISTRO.items()}
tabla_5_5 = pd.DataFrame([v['fila'] for v in REGISTRO.values()]).set_index('Modelo')
tabla_5_5['IC95% inf.'] = [np.percentile(BOOT[m][0], 2.5) for m in tabla_5_5.index]
tabla_5_5['IC95% sup.'] = [np.percentile(BOOT[m][0], 97.5) for m in tabla_5_5.index]
COLS = ['Tipo', 'Estrategia', 'Recall@10', 'IC95% inf.', 'IC95% sup.', 'Recall@1', 'Recall@5', 'Recall@20', 'MRR',
        'PR-AUC (pares)', 'ROC-AUC (pares)', 'Brier (pares)', 'Tiempo entrenamiento (s)',
        'Tiempo búsqueda hiperparámetros (s)', 'Inferencia por ficha (ms)', 'Brecha Recall@10 (train−test)']
tabla_5_5 = tabla_5_5[COLS].sort_values(['Recall@10', 'MRR'], ascending=False)
tabla_5_5.insert(0, 'Posición', np.arange(1, len(tabla_5_5) + 1))

def _resaltar(row):
    if row['Tipo'].startswith('Ensamble'):
        return ['background-color: #e8f1fb'] * len(row)
    if row.name == MEJOR_INDIVIDUAL:
        return ['background-color: #fff4d6'] * len(row)
    return [''] * len(row)

print("Tabla comparativa completa (azul: ensambles; amarillo: mejor modelo individual), ordenada por Recall@10:")
display(tabla_5_5.style.apply(_resaltar, axis=1).format(precision=3, na_rep='—'))

In [ ]:
# =============================================================================
# 5.5 (b) TABLA DE LA RÚBRICA (mejor individual + ensambles) Y DIFERENCIA PAREADA CONTRA EL MEJOR INDIVIDUAL
# =============================================================================
ENSAMBLES = [m for m, v in REGISTRO.items() if v['tipo'].startswith('Ensamble')]
FINALISTAS = [MEJOR_INDIVIDUAL] + ENSAMBLES
ref_r10, ref_mrr = BOOT[MEJOR_INDIVIDUAL]

filas = []
for m in FINALISTAS:
    d10 = BOOT[m][0] - ref_r10
    dmrr = BOOT[m][1] - ref_mrr
    lo10, hi10 = np.percentile(d10, [2.5, 97.5])
    filas.append({'Modelo': m, 'Tipo': REGISTRO[m]['tipo'],
                  'Recall@10': tabla_5_5.loc[m, 'Recall@10'],
                  'IC95%': f"[{tabla_5_5.loc[m, 'IC95% inf.']:.3f}, {tabla_5_5.loc[m, 'IC95% sup.']:.3f}]",
                  'MRR': tabla_5_5.loc[m, 'MRR'], 'Recall@1': tabla_5_5.loc[m, 'Recall@1'],
                  'PR-AUC (pares)': tabla_5_5.loc[m, 'PR-AUC (pares)'],
                  'Tiempo entrenamiento (s)': tabla_5_5.loc[m, 'Tiempo entrenamiento (s)'],
                  'Inferencia por ficha (ms)': tabla_5_5.loc[m, 'Inferencia por ficha (ms)'],
                  'Δ Recall@10 vs. mejor individual': d10.mean(),
                  'IC95% Δ': f"[{lo10:+.3f}, {hi10:+.3f}]" if m != MEJOR_INDIVIDUAL else '—',
                  'Δ MRR vs. mejor individual': dmrr.mean(),
                  '¿Mejora significativa?': ('—' if m == MEJOR_INDIVIDUAL else
                                             'Sí' if lo10 > 0 else 'Peor (significativo)' if hi10 < 0 else 'No (IC cruza 0)')})
tabla_rubrica = pd.DataFrame(filas).set_index('Modelo').sort_values(['Recall@10', 'MRR'], ascending=False)
print(f"Tabla de la rúbrica: mejor individual ({MEJOR_INDIVIDUAL}) + {len(ENSAMBLES)} ensambles, ordenada por Recall@10:")
display(tabla_rubrica.style.format(precision=3, na_rep='—'))

# --- Gráficas: diferencia pareada (forest plot) y desempeño vs. costo de entrenamiento
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
orden = [m for m in tabla_rubrica.index if m != MEJOR_INDIVIDUAL]
for i, m in enumerate(orden):
    d = BOOT[m][0] - ref_r10
    lo, hi = np.percentile(d, [2.5, 97.5])
    color = 'seagreen' if lo > 0 else ('firebrick' if hi < 0 else 'gray')
    ax[0].errorbar(d.mean(), i, xerr=[[d.mean() - lo], [hi - d.mean()]], fmt='o', capsize=6, ms=8, color=color)
ax[0].axvline(0, color='black', lw=1)
ax[0].set_yticks(range(len(orden))); ax[0].set_yticklabels(orden, fontsize=9); ax[0].invert_yaxis()
ax[0].set_xlabel(f'Δ Recall@10 vs. {MEJOR_INDIVIDUAL} (bootstrap pareado, IC 95 %)')
ax[0].set_title('¿Algún ensamble mejora significativamente al mejor individual?')

t_plot = tabla_5_5['Tiempo entrenamiento (s)'].clip(lower=1e-3)
for m in tabla_5_5.index:
    if m in PISOS:
        continue
    es_ens = REGISTRO[m]['tipo'].startswith('Ensamble')
    ax[1].scatter(t_plot[m], tabla_5_5.loc[m, 'Recall@10'], s=130 if es_ens else 70,
                  marker='D' if es_ens else 'o', color='royalblue' if es_ens else 'lightgray', edgecolor='black', zorder=3)
    if es_ens or m == MEJOR_INDIVIDUAL:
        ax[1].annotate(m, (t_plot[m], tabla_5_5.loc[m, 'Recall@10']), fontsize=8, xytext=(5, 4), textcoords='offset points')
ax[1].axhline(tabla_5_5.loc['Heurística TF-IDF (piso)', 'Recall@10'], ls='--', color='darkorange', label='Piso heurístico TF-IDF')
ax[1].axhline(TECHO_PROTOCOLO, ls=':', color='black', label=f'Techo del protocolo ({TECHO_PROTOCOLO:.3f})')
ax[1].set_xscale('log'); ax[1].set_xlabel('Tiempo de entrenamiento (s, escala log)'); ax[1].set_ylabel('Recall@10 (prueba)')
ax[1].set_title('Desempeño vs. costo de entrenamiento (rombos = ensambles)'); ax[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

print('CONCLUSIÓN AUTOMÁTICA (5.5):')
mejor_ens = max(ENSAMBLES, key=lambda m: (tabla_5_5.loc[m, 'Recall@10'], tabla_5_5.loc[m, 'MRR']))
signif = [m for m in ENSAMBLES if tabla_rubrica.loc[m, '¿Mejora significativa?'] == 'Sí']
print(f"  Mejor ensamble: {mejor_ens} (Recall@10 = {tabla_5_5.loc[mejor_ens, 'Recall@10']:.3f}, "
      f"MRR = {tabla_5_5.loc[mejor_ens, 'MRR']:.3f}) frente a {MEJOR_INDIVIDUAL} "
      f"(Recall@10 = {tabla_5_5.loc[MEJOR_INDIVIDUAL, 'Recall@10']:.3f}, MRR = {tabla_5_5.loc[MEJOR_INDIVIDUAL, 'MRR']:.3f}).")
if signif:
    print(f"  Ensambles con mejora estadísticamente significativa en Recall@10: {signif}.")
else:
    print("  Ningún ensamble mejora al mejor individual con significancia estadística (todos los IC de la diferencia "
          "pareada cruzan cero o quedan por debajo): con 107 fichas, las ganancias observadas caben en el ruido de muestreo.")
for m in ENSAMBLES:
    print(f"   - {m}: Δ Recall@10 = {tabla_rubrica.loc[m, 'Δ Recall@10 vs. mejor individual']:+.3f} "
          f"{tabla_rubrica.loc[m, 'IC95% Δ']}, Δ MRR = {tabla_rubrica.loc[m, 'Δ MRR vs. mejor individual']:+.3f}, "
          f"brecha train−test = {tabla_5_5.loc[m, 'Brecha Recall@10 (train−test)']:+.3f}")
alto_brecha = [m for m in ENSAMBLES if tabla_5_5.loc[m, 'Brecha Recall@10 (train−test)'] > 0.05]
if alto_brecha:
    print(f"  Señal de sobreajuste (brecha train−test > 5 pp): {alto_brecha}. Es el riesgo que motivó excluir "
          "ensambles en el Avance 4.")

## 5.6 Prueba de estrés de los finalistas

Las fichas reales llegan más incompletas que las del conjunto de prueba (nivel de ruido "medio"). Se regeneran las
pseudo-fichas de prueba con ruido **bajo, medio y alto** —mismas semillas que la prueba de estrés del Avance 4— y se
mide el Recall@10 del mejor individual, de cada ensamble y del piso heurístico. Son datos no vistos por ningún modelo
y alimentan el criterio de robustez de la matriz de decisión.

In [ ]:
# =============================================================================
# 5.6 PRUEBA DE ESTRÉS — ruido bajo / medio / alto sobre las fichas de prueba
# =============================================================================
filas = []
for nivel, sd in [('bajo', 101), ('medio', 102), ('alto', 103)]:
    Qn = generar_pseudofichas(ident[ident.ID_INTERNO.isin(ids_test)], nivel, seed=SEED + sd)
    Pn = construir_pares(Qn)
    scores = {'Heurística TF-IDF (piso)': score_b1(Pn).values}
    for m in FINALISTAS:
        scores[m] = puntuar(REGISTRO[m]['modelo'], Pn[FEATS_SEL])
    for m, s in scores.items():
        r = rangos_verdaderos(Pn.assign(_s=s), '_s', Qn.QUERY_ID.values)
        filas.append({'Ruido': nivel, 'Modelo': m, **metricas_ranking(r)})
estres = pd.DataFrame(filas)
tabla_estres = estres.pivot(index='Modelo', columns='Ruido', values='Recall@10')[['bajo', 'medio', 'alto']]
tabla_estres['caída bajo→alto'] = tabla_estres['bajo'] - tabla_estres['alto']
tabla_estres = tabla_estres.sort_values('alto', ascending=False)
display(tabla_estres.round(3))

mas_robusto = tabla_estres.drop(index='Heurística TF-IDF (piso)')['alto'].idxmax()
print('CONCLUSIÓN AUTOMÁTICA (5.6):')
print(f"  Con ruido alto, el finalista más robusto es {mas_robusto} (Recall@10 = {tabla_estres.loc[mas_robusto, 'alto']:.3f}); "
      f"el mejor individual queda en {tabla_estres.loc[MEJOR_INDIVIDUAL, 'alto']:.3f} y el piso heurístico en "
      f"{tabla_estres.loc['Heurística TF-IDF (piso)', 'alto']:.3f}.")
bajo_umbral = [m for m in FINALISTAS if tabla_estres.loc[m, 'alto'] < UMBRAL_NEGOCIO]
print(f"  Finalistas por debajo del umbral de negocio ({UMBRAL_NEGOCIO:.2f}) con ruido alto: {bajo_umbral or 'ninguno'}.")

## 5.7 Selección del modelo final

Se usa la **misma matriz de decisión del Avance 4**, ahora sobre el mejor individual y los seis ensambles. Mantener
los pesos evita ajustar el criterio al resultado.

| Criterio | Peso | Medición |
|---|---|---|
| Recall@10 (límite inferior del IC 95 %) | 30 % | Peor caso razonable del error más costoso |
| Recall@1 / MRR (promedio) | 15 % | Carga de revisión antes de llegar al correcto |
| Interpretabilidad | 25 % | Debido proceso: el perito y el Ministerio Público deben entender por qué se sugiere un candidato |
| Robustez al ruido | 15 % | Recall@10 con ruido alto (5.6) |
| Costo de inferencia | 10 % | ms por ficha |
| Calibración | 5 % | Brier a nivel par (antes se asignaba a juicio; ahora se mide) |

**Interpretabilidad (asignada con argumento, no medida):**

| Modelo | Puntaje | Argumento |
|---|---|---|
| Regresión logística | 5.0 | Un coeficiente por característica, interpretable como *odds ratio*; contribución por candidato directa |
| Bagging de regresión logística | 4.5 | Promedio de modelos lineales: los coeficientes promedio se leen igual, aunque la salida sea un promedio de probabilidades |
| Votación suave ponderada | 3.0 | Pesos explícitos por modelo; dentro de cada modelo la lectura depende de él (el MLP no es legible) |
| Stacking | 3.0 | El meta-modelo es lineal y dice cuánto confía en cada modelo base, pero no en cada característica |
| Blending | 3.0 | Igual que el stacking |
| Random Forest | 2.0 | Importancias globales; explicar un candidato concreto requiere técnicas post-hoc (SHAP) |
| Gradient Boosting | 2.0 | Igual que el Random Forest, con interacciones aún menos legibles |

Los criterios numéricos se reescalan a [1, 5] entre los siete candidatos (el mejor recibe 5 y el peor 1). La escala
**ordena, no mide**: una ventaja no significativa en Recall@10 puede verse grande en la matriz. Por eso la decisión
se lee junto con la columna "¿Mejora significativa?" de 5.5.

In [ ]:
# =============================================================================
# 5.7 MATRIZ DE DECISIÓN — mismo criterio y pesos que el Avance 4
# =============================================================================
INTERPRETABILIDAD = {'Regresión logística': 5.0, 'Bagging de regresión logística': 4.5,
                     'Votación suave ponderada': 3.0, 'Stacking': 3.0, 'Blending': 3.0,
                     'Random Forest': 2.0, 'Gradient Boosting (HGB)': 2.0,
                     # por si el mejor individual de la ejecución no es la regresión logística:
                     'SVM lineal (ajustado)': 4.0, 'SVM lineal': 4.0, 'LDA': 4.0, 'Naive Bayes gaussiano': 3.5,
                     'Árbol de decisión (CART)': 3.5, 'MLP (ajustado)': 2.0, 'Perceptrón multicapa (MLP)': 2.0,
                     'k-NN': 2.0, 'SVM RBF escalable (Nystroem)': 1.5}

def escalar_1a5(valor, peor, mejor):
    if peor == mejor or pd.isna(valor):
        return 3.0 if peor == mejor else 1.0
    return float(np.clip(1 + 4 * (valor - peor) / (mejor - peor), 1, 5))

CAND = FINALISTAS
crit = pd.DataFrame({
    'r10_ic_inf': {m: tabla_5_5.loc[m, 'IC95% inf.'] for m in CAND},
    'r1_mrr':     {m: (tabla_5_5.loc[m, 'Recall@1'] + tabla_5_5.loc[m, 'MRR']) / 2 for m in CAND},
    'robustez':   {m: tabla_estres.loc[m, 'alto'] for m in CAND},
    'costo':      {m: tabla_5_5.loc[m, 'Inferencia por ficha (ms)'] for m in CAND},
    'brier':      {m: tabla_5_5.loc[m, 'Brier (pares)'] for m in CAND},
})
def esc(col, mayor_es_mejor=True):
    v = crit[col]
    peor, mejor = (v.min(), v.max()) if mayor_es_mejor else (v.max(), v.min())
    return {m: escalar_1a5(v[m], peor, mejor) for m in CAND}

matriz = pd.DataFrame({
    'Recall@10 (IC inf.)': esc('r10_ic_inf'),
    'Recall@1 / MRR':      esc('r1_mrr'),
    'Interpretabilidad':   {m: INTERPRETABILIDAD.get(m, 2.0) for m in CAND},
    'Robustez al ruido':   esc('robustez'),
    'Costo de inferencia': esc('costo', mayor_es_mejor=False),
    'Calibración':         esc('brier', mayor_es_mejor=False),
}).loc[CAND]
PESOS_DECISION = pd.Series({'Recall@10 (IC inf.)': 0.30, 'Recall@1 / MRR': 0.15, 'Interpretabilidad': 0.25,
                            'Robustez al ruido': 0.15, 'Costo de inferencia': 0.10, 'Calibración': 0.05})
matriz['Puntaje ponderado'] = (matriz[PESOS_DECISION.index] * PESOS_DECISION).sum(axis=1)
matriz['¿Mejora significativa vs. mejor individual?'] = [tabla_rubrica.loc[m, '¿Mejora significativa?'] for m in CAND]
matriz = matriz.sort_values('Puntaje ponderado', ascending=False)
display(crit.rename(columns={'r10_ic_inf': 'Recall@10 IC inf.', 'r1_mrr': '(Recall@1+MRR)/2',
                             'robustez': 'Recall@10 ruido alto', 'costo': 'ms/ficha', 'brier': 'Brier'}).round(4))
display(matriz.style.format(precision=2).background_gradient(subset=['Puntaje ponderado'], cmap='Greens'))

# --- Regla de decisión
#  1) Gana el mayor puntaje ponderado.
#  2) Salvaguarda de negocio: el modelo final debe superar el umbral de negocio en su IC inferior y con ruido alto.
#  3) Si el ganador es un ensamble SIN mejora significativa y la diferencia de puntaje con el mejor individual es
#     menor a 0.25 puntos, se prefiere el modelo más simple (navaja de Occam): la complejidad adicional no está
#     justificada por evidencia.
ganador = matriz.index[0]
for m in matriz.index:
    if tabla_5_5.loc[m, 'IC95% inf.'] >= UMBRAL_NEGOCIO and tabla_estres.loc[m, 'alto'] >= UMBRAL_NEGOCIO:
        ganador = m
        break
regla_occam = (ganador != MEJOR_INDIVIDUAL
               and tabla_rubrica.loc[ganador, '¿Mejora significativa?'] != 'Sí'
               and matriz.loc[ganador, 'Puntaje ponderado'] - matriz.loc[MEJOR_INDIVIDUAL, 'Puntaje ponderado'] < 0.25)
MODELO_FINAL_NOMBRE = MEJOR_INDIVIDUAL if regla_occam else ganador
MODELO_FINAL = REGISTRO[MODELO_FINAL_NOMBRE]['modelo']
S_FINAL = REGISTRO[MODELO_FINAL_NOMBRE]['s_test']
R_FINAL = REGISTRO[MODELO_FINAL_NOMBRE]['r_test']
# Modelo de comparación en los gráficos: si el final es un ensamble, el mejor individual; si no, el mejor ensamble
MODELO_COMP_NOMBRE = MEJOR_INDIVIDUAL if MODELO_FINAL_NOMBRE != MEJOR_INDIVIDUAL else mejor_ens

print(f"\nMODELO FINAL ELEGIDO: {MODELO_FINAL_NOMBRE}")
if regla_occam:
    print(f"  (El mayor puntaje fue de {ganador}, pero su ventaja sobre {MEJOR_INDIVIDUAL} no es significativa y la "
          f"diferencia de puntaje es < 0.25: se aplica la regla de parsimonia.)")
f = tabla_5_5.loc[MODELO_FINAL_NOMBRE]
print(f"  Recall@10 = {f['Recall@10']:.3f} [{f['IC95% inf.']:.3f}, {f['IC95% sup.']:.3f}] | MRR = {f['MRR']:.3f} | "
      f"Recall@1 = {f['Recall@1']:.3f} | PR-AUC = {f['PR-AUC (pares)']:.3f} | ruido alto = {tabla_estres.loc[MODELO_FINAL_NOMBRE, 'alto']:.3f}")
print(f"  Entrenamiento = {f['Tiempo entrenamiento (s)']:.2f} s | inferencia = {f['Inferencia por ficha (ms)']:.3f} ms por ficha")
print(f"  Umbral de negocio ({UMBRAL_NEGOCIO:.2f}): {'CUMPLE' if f['IC95% inf.'] >= UMBRAL_NEGOCIO else 'NO CUMPLE'} en el "
      f"IC inferior. Techo del protocolo: {TECHO_PROTOCOLO:.3f} (margen restante {TECHO_PROTOCOLO - f['Recall@10']:.3f}).")

### Argumentos de negocio para la elección

La celda anterior aplica la regla; estos son los argumentos que la sostienen, en el orden en que pesan para la
Dirección de Servicios Periciales:

1. **El falso negativo manda.** Un expediente correcto fuera de la lista corta no se revisa. Por eso el criterio de
   mayor peso es el límite **inferior** del IC de Recall@10, no el valor puntual: se elige pensando en el peor caso
   razonable.
2. **La sugerencia debe poder explicarse.** El perito firma un dictamen y el Ministerio Público puede pedir por qué
   se revisó un expediente y no otro. Un modelo que gana por una diferencia no significativa pero no puede explicar
   un candidato concreto es un mal intercambio en un proceso con consecuencias legales.
3. **Las fichas reales vienen incompletas.** La robustez con ruido alto aproxima mejor la operación real que el
   conjunto de prueba con ruido medio.
4. **El costo no es la restricción.** Todos los finalistas puntúan una ficha en milisegundos sin GPU; el criterio de
   costo pesa poco a propósito.
5. **Parsimonia.** Si un ensamble no mejora al mejor individual con significancia, la complejidad adicional
   (reentrenar varios modelos, monitorear varias fuentes de error) no está justificada.

## 5.8 Gráficos del modelo final

Ocho gráficos sobre el conjunto de prueba (datos no vistos). Cada uno compara el modelo final con el modelo de
referencia (`MODELO_COMP_NOMBRE`: el mejor individual si el final es un ensamble, o el mejor ensamble si el final es
individual) y termina con una **interpretación automática** calculada con los resultados de la ejecución.

| # | Gráfico | Pregunta que responde |
|---|---|---|
| 1 | Curva Recall@k | ¿Cuántos candidatos debe revisar el perito para encontrar al correcto? |
| 2 | Curva Precisión-Recall (pares) | ¿Qué tan limpia es la lista con 0.16 % de positivos? |
| 3 | Curva ROC (pares) | ¿Qué tan bien separa pares correctos de incorrectos en general? |
| 4 | Matriz de confusión operativa (regla top-10) | ¿Cuánta carga de revisión genera y cuántas coincidencias se pierden? |
| 5 | Importancia de características (permutación) | ¿Qué información usa el modelo para ordenar? |
| 6 | Análisis de residuos (rango del correcto y errores por segmento) | ¿Dónde y por qué falla? |
| 7 | Calibración y distribución de scores | ¿Se puede mostrar el score al perito como "confianza"? |
| 8 | Robustez al ruido | ¿Cómo se degrada con fichas más incompletas? |

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 1 — CURVA RECALL@k (k = 1..50)
# =============================================================================
S_COMP, R_COMP = REGISTRO[MODELO_COMP_NOMBRE]['s_test'], REGISTRO[MODELO_COMP_NOMBRE]['r_test']
R_PISO = REGISTRO['Heurística TF-IDF (piso)']['r_test']
ks = np.arange(1, 51)
curvas = {MODELO_FINAL_NOMBRE + ' (final)': R_FINAL, MODELO_COMP_NOMBRE: R_COMP, 'Heurística TF-IDF (piso)': R_PISO}

fig, ax = plt.subplots(figsize=(10, 5.5))
for (nombre, r), (c, ls, lw) in zip(curvas.items(), [('seagreen', '-', 3), ('royalblue', '-', 1.8), ('darkorange', '--', 1.8)]):
    ax.plot(ks, [np.mean(r <= k) for k in ks], ls=ls, lw=lw, color=c, label=nombre)
ax.axhline(TECHO_PROTOCOLO, ls=':', color='black', label=f'Techo del protocolo ({TECHO_PROTOCOLO:.3f})')
ax.axvline(10, ls=':', color='gray'); ax.text(10.5, 0.42, 'k = 10\n(lista corta del perito)', color='gray', fontsize=8)
ax.set_xlabel('k (candidatos revisados por ficha)'); ax.set_ylabel('Recall@k (fracción de fichas con el correcto en el top-k)')
ax.set_title(f'Gráfico 1. Curva Recall@k en prueba — {MODELO_FINAL_NOMBRE}'); ax.set_ylim(0.4, 1.01); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

rk = {k: np.mean(R_FINAL <= k) for k in (1, 5, 10, 20, 50)}
k90 = next((k for k in range(1, 501) if np.mean(R_FINAL <= k) >= 0.9 * TECHO_PROTOCOLO), None)
print('INTERPRETACIÓN (Gráfico 1):')
print(f"  - El modelo final pone el expediente correcto en 1.º lugar en {rk[1]:.1%} de las fichas, en el top-5 en {rk[5]:.1%}, "
      f"en el top-10 en {rk[10]:.1%} y en el top-20 en {rk[20]:.1%}.")
print(f"  - Para encontrar al correcto en el 90 % de las fichas alcanzables (90 % del techo), el perito necesita revisar "
      f"{k90 if k90 else '>500'} candidatos por ficha, de ~{len(P_test) / len(Q_test):.0f} que deja el blocking.")
print(f"  - La curva casi plana después de k=10 ({rk[10]:.3f} -> {rk[50]:.3f} en k=50) indica que los fallos no son "
      "'casi aciertos': cuando el modelo no encuentra al correcto en los primeros lugares, suele quedar muy abajo. "
      "El límite es la información de la ficha, no el tamaño de la lista.")
d = {k: np.mean(R_FINAL <= k) - np.mean(R_COMP <= k) for k in (1, 10, 20)}
print(f"  - Frente a {MODELO_COMP_NOMBRE}: Δ Recall@1 = {d[1]:+.3f}, Δ Recall@10 = {d[10]:+.3f}, Δ Recall@20 = {d[20]:+.3f}.")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 2 — CURVA PRECISIÓN-RECALL A NIVEL PAR (con el punto de operación top-10)
# =============================================================================
def top_k_mask(P, s, k=10):
    """1 si el par queda dentro del top-k de su ficha (empates pesimistas: rango = 1 + # con score mayor o igual)."""
    rk = P.assign(_s=s).groupby('QUERY_ID')['_s'].rank(method='max', ascending=False)
    return (rk <= k).astype(int).values

pred_final = top_k_mask(P_test, S_FINAL)
pred_comp = top_k_mask(P_test, S_COMP)
prev = P_test.y.mean()

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for s, nombre, c in [(S_FINAL, MODELO_FINAL_NOMBRE + ' (final)', 'seagreen'), (S_COMP, MODELO_COMP_NOMBRE, 'royalblue')]:
    p, r, _ = precision_recall_curve(P_test.y, s)
    ax.plot(r, p, lw=2.5 if 'final' in nombre else 1.6, color=c, label=f"{nombre} (AP = {average_precision_score(P_test.y, s):.3f})")
op_r = pred_final[P_test.y.values == 1].mean(); op_p = P_test.y.values[pred_final == 1].mean()
ax.scatter(op_r, op_p, s=140, marker='*', color='black', zorder=5, label=f'Punto de operación top-10 (P={op_p:.3f}, R={op_r:.3f})')
ax.axhline(prev, ls='--', color='gray', label=f'Azar = prevalencia ({prev:.4f})')
ax.set_xlabel('Recall a nivel par'); ax.set_ylabel('Precisión a nivel par'); ax.set_ylim(0, 1.02)
ax.set_title('Gráfico 2. Curva Precisión-Recall (pares de prueba)'); ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

ap_f = average_precision_score(P_test.y, S_FINAL)
print('INTERPRETACIÓN (Gráfico 2):')
print(f"  - AP = {ap_f:.3f} frente a una prevalencia de {prev:.4f}: el modelo concentra los pares correctos "
      f"{ap_f / prev:.0f} veces mejor que el azar.")
print(f"  - La curva cae rápido a la derecha: los últimos pares correctos solo se recuperan aceptando miles de pares "
      "incorrectos. Son las fichas con poca información (ver Gráfico 6).")
print(f"  - En el punto de operación real (revisar el top-10 de cada ficha) la precisión a nivel par es {op_p:.3f}: "
      f"aproximadamente 1 de cada {1 / op_p:.0f} candidatos revisados es el correcto. Es el costo esperado del diseño "
      "'el modelo sugiere, el perito confirma', y es mucho menor que revisar los ~"
      f"{len(P_test) / len(Q_test):.0f} candidatos del bloque completo.")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 3 — CURVA ROC A NIVEL PAR
# =============================================================================
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for a, xscale in zip(ax, ['linear', 'log']):
    for s, nombre, c in [(S_FINAL, MODELO_FINAL_NOMBRE + ' (final)', 'seagreen'), (S_COMP, MODELO_COMP_NOMBRE, 'royalblue')]:
        fpr, tpr, _ = roc_curve(P_test.y, s)
        a.plot(fpr, tpr, lw=2.5 if 'final' in nombre else 1.6, color=c, label=f"{nombre} (AUC = {roc_auc_score(P_test.y, s):.3f})")
    a.plot([1e-5, 1], [1e-5, 1], ls='--', color='gray', label='Azar')
    a.set_xscale(xscale); a.set_xlabel('Tasa de falsos positivos' + (' (escala log)' if xscale == 'log' else ''))
    a.set_ylabel('Tasa de verdaderos positivos'); a.legend(fontsize=8, loc='lower right')
    if xscale == 'log':
        a.set_xlim(1e-4, 1)
ax[0].set_title('Gráfico 3a. Curva ROC (pares de prueba)')
ax[1].set_title('Gráfico 3b. Misma curva con FPR en escala log (zona útil para el perito)')
plt.tight_layout(); plt.show()

fpr, tpr, _ = roc_curve(P_test.y, S_FINAL)
fpr90 = fpr[np.searchsorted(tpr, 0.90)] if (tpr >= 0.90).any() else np.nan
n_neg_ficha = (P_test.y == 0).sum() / len(Q_test)
print('INTERPRETACIÓN (Gráfico 3):')
print(f"  - ROC-AUC = {roc_auc_score(P_test.y, S_FINAL):.3f}: un par correcto elegido al azar recibe un score mayor que uno "
      f"incorrecto en {roc_auc_score(P_test.y, S_FINAL):.1%} de los casos.")
print(f"  - El valor alto es engañoso con 0.16 % de positivos: para recuperar 90 % de los pares correctos la FPR es "
      f"{fpr90:.4f}, que equivale a ~{fpr90 * n_neg_ficha:.0f} candidatos incorrectos por ficha. Por eso la escala log "
      "(3b) y la curva PR (Gráfico 2) son las lecturas relevantes, y el ROC-AUC solo se reporta como complemento "
      "(Saito y Rehmsmeier, 2015).")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 4 — MATRIZ DE CONFUSIÓN OPERATIVA (regla: se envían a revisión los 10 primeros de cada ficha)
# =============================================================================
from matplotlib.colors import LogNorm
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, pred, nombre in [(ax[0], pred_final, MODELO_FINAL_NOMBRE + ' (final)'), (ax[1], pred_comp, MODELO_COMP_NOMBRE)]:
    cm = confusion_matrix(P_test.y, pred)
    etiquetas = np.array([[f"{v:,}\n({v / cm[i].sum():.2%} de la fila)" for v in row] for i, row in enumerate(cm)])
    sns.heatmap(cm, annot=etiquetas, fmt='', cmap='Greens', cbar=False, ax=a, norm=LogNorm(),
                xticklabels=['No revisar', 'Revisar (top-10)'], yticklabels=['Par incorrecto', 'Par correcto'])
    a.set_xlabel('Decisión del sistema'); a.set_ylabel('Realidad'); a.set_title(f'Gráfico 4. Matriz de confusión — {nombre}', fontsize=10)

# Nivel ficha: ¿se encontró al correcto en el top-10? (incluye las fichas perdidas por el blocking)
ficha_ok = (R_FINAL <= 10)
perdida_block = ~np.isfinite(R_FINAL)
cat = pd.Series(np.where(ficha_ok, 'Correcto\nen top-10', np.where(perdida_block, 'Perdida\nen el blocking', 'Correcto\nfuera del top-10')))
cat.value_counts().reindex(['Correcto\nen top-10', 'Correcto\nfuera del top-10', 'Perdida\nen el blocking']).fillna(0).plot(
    kind='bar', ax=ax[2], color=['seagreen', 'firebrick', 'gray'], edgecolor='black')
for i, v in enumerate(cat.value_counts().reindex(['Correcto\nen top-10', 'Correcto\nfuera del top-10', 'Perdida\nen el blocking']).fillna(0)):
    ax[2].text(i, v + 0.5, f'{int(v)}', ha='center')
ax[2].set_title(f'Resultado por ficha (n = {len(Q_test)})', fontsize=10); ax[2].set_ylabel('Fichas'); ax[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

tn, fp, fn, tp = confusion_matrix(P_test.y, pred_final).ravel()
_, fp_c, fn_c, tp_c = confusion_matrix(P_test.y, pred_comp).ravel()
print('INTERPRETACIÓN (Gráfico 4):')
print(f"  - De {tp + fn} pares correctos que sobrevivieron al blocking, el modelo final envía {tp} a revisión (VP) y deja "
      f"fuera {fn} (FN). A esos se suman {perdida_block.sum()} fichas cuyo correcto ya había eliminado el blocking.")
print(f"  - Carga de revisión: {tp + fp:,} pares enviados para {len(Q_test)} fichas (≤ 10 por ficha), de los cuales "
      f"{fp:,} son falsos positivos. Precisión operativa = {tp / (tp + fp):.3f}.")
print(f"  - Se descartan automáticamente {tn:,} de {tn + fp:,} pares incorrectos ({tn / (tn + fp):.1%}): es el ahorro de "
      "trabajo pericial frente a revisar el bloque completo.")
print(f"  - Frente a {MODELO_COMP_NOMBRE}: {tp - tp_c:+d} verdaderos positivos y {fn - fn_c:+d} falsos negativos con la "
      "misma carga de revisión (la regla top-10 fija el número de candidatos por ficha).")
print("  - Los falsos negativos son el error que importa: cada uno es una persona cuyo expediente no se revisaría.")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 5 — IMPORTANCIA DE CARACTERÍSTICAS POR PERMUTACIÓN (caída de Recall@10 y MRR en prueba)
# =============================================================================
# Permutación agnóstica al modelo: sirve igual para un modelo lineal, un bosque o un ensamble heterogéneo.
N_REP = 5
def importancia_permutacion(modelo, n_rep=N_REP):
    base = metricas_ranking(rangos_rapidos(P_test.QUERY_ID.values, P_test.y.values, puntuar(modelo, X_TE), Q_test.QUERY_ID.values))
    rng = np.random.default_rng(SEED)
    filas = []
    for f in FEATS_SEL:
        d10, dmrr = [], []
        for _ in range(n_rep):
            Xp = X_TE.copy(); Xp[f] = rng.permutation(Xp[f].values)
            m = metricas_ranking(rangos_rapidos(P_test.QUERY_ID.values, P_test.y.values, puntuar(modelo, Xp), Q_test.QUERY_ID.values))
            d10.append(base['Recall@10'] - m['Recall@10']); dmrr.append(base['MRR'] - m['MRR'])
        filas.append({'Característica': f, 'Δ Recall@10': np.mean(d10), 'DE Δ Recall@10': np.std(d10),
                      'Δ MRR': np.mean(dmrr), 'DE Δ MRR': np.std(dmrr)})
    return pd.DataFrame(filas).set_index('Característica').sort_values('Δ MRR', ascending=False)

imp_final = importancia_permutacion(MODELO_FINAL)
imp_comp = importancia_permutacion(REGISTRO[MODELO_COMP_NOMBRE]['modelo'])

fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
orden = imp_final.index[::-1]
y_pos = np.arange(len(orden))
for a, met in zip(ax, ['Δ Recall@10', 'Δ MRR']):
    a.barh(y_pos + 0.2, imp_final.loc[orden, met], 0.4, xerr=imp_final.loc[orden, 'DE ' + met], color='seagreen',
           edgecolor='black', label=MODELO_FINAL_NOMBRE + ' (final)')
    a.barh(y_pos - 0.2, imp_comp.loc[orden, met], 0.4, xerr=imp_comp.loc[orden, 'DE ' + met], color='lightsteelblue',
           edgecolor='black', label=MODELO_COMP_NOMBRE)
    a.set_yticks(y_pos); a.set_yticklabels(orden); a.axvline(0, color='black', lw=0.8)
    a.set_xlabel(f'{met} al permutar la característica (mayor = más importante)'); a.legend(fontsize=8, loc='lower right')
ax[0].set_title('Gráfico 5a. Importancia por permutación — Recall@10')
ax[1].set_title('Gráfico 5b. Importancia por permutación — MRR')
plt.tight_layout(); plt.show()

top3 = list(imp_final.index[:3])
nulas = list(imp_final.index[(imp_final['Δ MRR'].abs() < 0.005) & (imp_final['Δ Recall@10'].abs() < 0.005)])
print('INTERPRETACIÓN (Gráfico 5):')
print(f"  - Las tres características que más sostienen el ranking del modelo final son {top3}: al desordenar "
      f"'{top3[0]}' el MRR cae {imp_final.loc[top3[0], 'Δ MRR']:.3f} y el Recall@10 {imp_final.loc[top3[0], 'Δ Recall@10']:.3f}.")
print(f"  - Características casi irrelevantes para este modelo (|Δ| < 0.005 en ambas métricas): {nulas or 'ninguna'}.")
rho = imp_final['Δ MRR'].corr(imp_comp['Δ MRR'], method='spearman')
print(f"  - Correlación de Spearman entre la importancia del modelo final y la de {MODELO_COMP_NOMBRE}: {rho:.2f}. "
      f"{'Ambos usan la información de forma parecida: la ganancia del ensamble no viene de descubrir variables nuevas.' if rho > 0.7 else 'Los modelos ponderan la información de forma distinta: ahí está su complementariedad.'}")
print("  - Coincide con el Avance 4: las similitudes de texto de señas y ropa concentran la señal. Mejorar el sistema "
      "pasa por representar mejor esos textos (embeddings semánticos), no por más capacidad del modelo.")
if isinstance(MODELO_FINAL, StackingClassifier):
    print("  - Pesos del meta-modelo del stacking:", dict(zip(nombres_meta, np.round(coef_meta, 3))))

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 6 — ANÁLISIS DE RESIDUOS: rango del expediente correcto y errores por tipo de ficha
# =============================================================================
seg = pd.DataFrame({
    'Sin señas en la ficha': Q_test.SENAS.eq('').values,
    'Sin ropa en la ficha': Q_test.ROPA.eq('').values,
    'Sin estatura': Q_test.ESTATURA.isna().values,
    'Sin edad': Q_test.EDAD.isna().values,
})
seg['Con señas y ropa'] = ~seg['Sin señas en la ficha'] & ~seg['Sin ropa en la ficha']

fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
r_plot = np.where(np.isfinite(R_FINAL), R_FINAL, np.nan)
bins = np.unique(np.logspace(0, np.log10(max(np.nanmax(r_plot), 11) + 1), 25).astype(int))
ax[0].hist(r_plot[~np.isnan(r_plot)], bins=bins, color='seagreen', edgecolor='black')
ax[0].axvline(10.5, ls='--', color='firebrick', label='Corte top-10')
ax[0].set_xscale('log'); ax[0].set_xlabel('Rango del expediente correcto (1 = primero; escala log)')
ax[0].set_ylabel('Fichas'); ax[0].set_title(f'Gráfico 6a. Distribución del rango del correcto — {MODELO_FINAL_NOMBRE}')
ax[0].legend()

filas = []
for s_name in seg.columns:
    msk = seg[s_name].values
    if msk.sum() == 0:
        continue
    filas.append({'Segmento': f"{s_name} (n={msk.sum()})",
                  MODELO_FINAL_NOMBRE + ' (final)': np.mean(R_FINAL[msk] <= 10),
                  MODELO_COMP_NOMBRE: np.mean(R_COMP[msk] <= 10)})
tabla_seg = pd.DataFrame(filas).set_index('Segmento')
tabla_seg.plot(kind='barh', ax=ax[1], color=['seagreen', 'lightsteelblue'], edgecolor='black')
ax[1].axvline(np.mean(R_FINAL <= 10), ls='--', color='seagreen', lw=1, label='Recall@10 global (final)')
ax[1].set_xlim(0, 1.05); ax[1].set_xlabel('Recall@10 dentro del segmento')
ax[1].set_title('Gráfico 6b. Recall@10 por tipo de ficha (residuos por segmento)'); ax[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

fallos = (R_FINAL > 10)
print('INTERPRETACIÓN (Gráfico 6):')
print(f"  - {fallos.sum()} de {len(Q_test)} fichas quedan fuera del top-10. Mediana del rango en esos fallos: "
      f"{np.nanmedian(np.where(np.isfinite(R_FINAL[fallos]), R_FINAL[fallos], np.nan)) if fallos.any() else float('nan'):.0f} "
      f"({perdida_block.sum()} perdidas en el blocking, rango infinito).")
display(tabla_seg.round(3))
peor = tabla_seg[MODELO_FINAL_NOMBRE + ' (final)'].idxmin()
print(f"  - El segmento más difícil es '{peor}' (Recall@10 = {tabla_seg.loc[peor, MODELO_FINAL_NOMBRE + ' (final)']:.3f}). "
      "Sin texto que comparar, el modelo depende de edad, estatura y municipio, que no individualizan.")
tasa_sin_senas = seg['Sin señas en la ficha'][fallos].mean() if fallos.any() else np.nan
print(f"  - {tasa_sin_senas:.0%} de los fallos son fichas sin señas particulares, frente a "
      f"{seg['Sin señas en la ficha'].mean():.0%} del total: los residuos no son aleatorios, se concentran donde falta "
      "información. Recomendación operativa: priorizar la captura de señas particulares en ficha y necropsia.")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 7 — CALIBRACIÓN Y DISTRIBUCIÓN DE SCORES
# =============================================================================
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
s_plot_final = S_FINAL if REGISTRO[MODELO_FINAL_NOMBRE]['es_prob'] else 1 / (1 + np.exp(-S_FINAL))
for s, nombre, c, es_prob in [(s_plot_final, MODELO_FINAL_NOMBRE + ' (final)', 'seagreen', REGISTRO[MODELO_FINAL_NOMBRE]['es_prob']),
                               (S_COMP, MODELO_COMP_NOMBRE, 'royalblue', REGISTRO[MODELO_COMP_NOMBRE]['es_prob'])]:
    if not es_prob:
        continue
    frac, media = calibration_curve(P_test.y, s, n_bins=10, strategy='quantile')
    ax[0].plot(media, frac, 'o-', color=c, lw=2, label=f"{nombre} (Brier = {brier_score_loss(P_test.y, s):.5f})")
# Con 0.16 % de positivos las probabilidades viven cerca de 0: se hace zoom a la zona con datos
lim = 1.15 * max([l.get_xdata().max() for l in ax[0].lines] + [l.get_ydata().max() for l in ax[0].lines] + [1e-3])
ax[0].plot([0, lim], [0, lim], ls='--', color='gray', label='Calibración perfecta')
ax[0].set_xlim(0, lim); ax[0].set_ylim(0, lim)
ax[0].set_xlabel('Probabilidad media predicha (por decil de score)'); ax[0].set_ylabel('Fracción real de pares correctos')
ax[0].set_title('Gráfico 7a. Curva de calibración (pares de prueba)'); ax[0].legend(fontsize=8, loc='upper left')

pos, neg = s_plot_final[P_test.y.values == 1], s_plot_final[P_test.y.values == 0]
b = np.logspace(np.log10(max(s_plot_final.min(), 1e-8)), np.log10(s_plot_final.max() + 1e-12), 50)
ax[1].hist(neg, bins=b, alpha=0.6, color='gray', label=f'Pares incorrectos (n={len(neg):,})', weights=np.ones(len(neg)) / len(neg))
ax[1].hist(pos, bins=b, alpha=0.75, color='seagreen', label=f'Pares correctos (n={len(pos):,})', weights=np.ones(len(pos)) / len(pos))
ax[1].set_xscale('log'); ax[1].set_xlabel('Score del modelo final (escala log)'); ax[1].set_ylabel('Proporción de los pares de cada clase')
ax[1].set_title('Gráfico 7b. Distribución del score por clase'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print('INTERPRETACIÓN (Gráfico 7):')
if REGISTRO[MODELO_FINAL_NOMBRE]['es_prob']:
    frac, media = calibration_curve(P_test.y, S_FINAL, n_bins=10, strategy='quantile')
    razon = (frac[-1] / media[-1]) if media[-1] > 0 else np.nan
    print(f"  - Brier = {brier_score_loss(P_test.y, S_FINAL):.5f}. En el decil de score más alto el modelo predice en "
          f"promedio {media[-1]:.3f} y la frecuencia real es {frac[-1]:.3f} (razón real/predicho = {razon:.2f}). "
          f"{'Está razonablemente calibrado: el score puede presentarse como probabilidad.' if 0.5 <= razon <= 2 else 'No está calibrado: el score ordena bien, pero NO debe mostrarse al perito como probabilidad sin recalibrarlo (Platt o isotónica).'}")
else:
    print("  - El modelo final no produce probabilidades: el score solo ordena y no debe mostrarse como confianza.")
print(f"  - Mediana del score: pares correctos = {np.median(pos):.4g} vs. incorrectos = {np.median(neg):.4g} "
      f"({np.median(pos) / max(np.median(neg), 1e-12):.0f} veces mayor). El traslape entre ambas distribuciones explica "
      "por qué no existe un umbral absoluto útil: por eso la decisión se toma por ranking dentro de cada ficha (top-10).")

In [ ]:
# =============================================================================
# 5.8 GRÁFICO 8 — ROBUSTEZ AL RUIDO DE LA FICHA (finalistas)
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 5.5))
paleta = dict(zip(tabla_estres.index, sns.color_palette('tab10', n_colors=len(tabla_estres))))
for m, g in estres.groupby('Modelo'):
    g = g.set_index('Ruido').loc[['bajo', 'medio', 'alto']]
    es_final = m == MODELO_FINAL_NOMBRE
    ax.plot(g.index, g['Recall@10'], 'o-', lw=3.5 if es_final else 1.4, ms=8 if es_final else 5,
            ls='--' if m in PISOS else '-', color='black' if es_final else paleta[m],
            label=m + (' (final)' if es_final else ''), zorder=5 if es_final else 2)
ax.axhline(UMBRAL_NEGOCIO, ls=':', color='crimson', label=f'Umbral de negocio ({UMBRAL_NEGOCIO:.2f})')
ax.set_ylim(0.4, 1.0); ax.set_xlabel('Nivel de ruido de la pseudo-ficha'); ax.set_ylabel('Recall@10')
ax.set_title('Gráfico 8. Robustez: Recall@10 según la calidad de la ficha'); ax.legend(fontsize=7, loc='lower left', ncol=2)
plt.tight_layout(); plt.show()

fila = tabla_estres.loc[MODELO_FINAL_NOMBRE]
print('INTERPRETACIÓN (Gráfico 8):')
print(f"  - El modelo final pasa de {fila['bajo']:.3f} con ruido bajo a {fila['alto']:.3f} con ruido alto "
      f"(caída de {fila['caída bajo→alto']:.3f}). {'Se mantiene sobre' if fila['alto'] >= UMBRAL_NEGOCIO else 'Cae por debajo de'} "
      f"el umbral de negocio ({UMBRAL_NEGOCIO:.2f}) en el peor escenario simulado.")
rank_alto = tabla_estres.drop(index='Heurística TF-IDF (piso)')['alto'].rank(ascending=False)
print(f"  - Con ruido alto ocupa el lugar {int(rank_alto[MODELO_FINAL_NOMBRE])} de {len(rank_alto)} finalistas. "
      f"La dispersión entre finalistas con ruido alto es de {tabla_estres['alto'].max() - tabla_estres['alto'].min():.3f}: "
      "cuando la ficha pierde información, todos los modelos convergen porque ninguno puede recuperar lo que la ficha no dice.")

## 5.9 Demostración para el perito

Las dos fichas sintéticas del Avance 4 (`FICHA-DEMO-001`, rica en datos, y `FICHA-DEMO-002`, sin señas
particulares) se pasan por el modelo final. No corresponden a personas reales. La salida es el top-5 de candidatos
con las variables que el perito usaría para decidir qué expediente abrir primero. Si el modelo final es un ensamble
heterogéneo, se muestran además los scores de cada modelo base: es su forma de "justificar" la posición.

In [ ]:
# =============================================================================
# 5.9 DEMO — ranking del modelo final para dos fichas sintéticas
# =============================================================================
ref_pos = pd.Series(S_FINAL[P_test.y.values == 1])
print(f"Modelo final: {MODELO_FINAL_NOMBRE}")
print(f"Referencia: score de las coincidencias verdaderas en prueba -> mediana {ref_pos.median():.4g}, percentil 10 {ref_pos.quantile(0.10):.4g}\n")

for _, f0 in ficha.iterrows():
    Q_demo = pd.DataFrame([dict(QUERY_ID=-1, ID_VERDADERO=-1, SEXO=parse_sexo(f0['PRE_SEXO']), EDAD=float(f0['PRE_EDAD']),
                                ESTATURA=float(f0['PRE_ESTATURA']), MUNICIPIO=limpiar_texto(f0['PRE_MUNICIPIO']),
                                SENAS=limpiar_texto(f0['PRE_SENAS']), ROPA=limpiar_texto(f0['PRE_ROPA']))])
    P_demo = construir_pares(Q_demo)
    P_demo['score'] = puntuar(MODELO_FINAL, P_demo[FEATS_SEL])
    if isinstance(MODELO_FINAL, (StackingClassifier, Blending)):
        bases = MODELO_FINAL.named_estimators_ if isinstance(MODELO_FINAL, StackingClassifier) else MODELO_FINAL.bases_
        for k, m in bases.items():
            P_demo[f'score base: {k}'] = puntuar(m, P_demo[FEATS_SEL])
    senas_txt = 'sin señas registradas' if limpiar_texto(f0['PRE_SENAS']) == '' else 'con señas registradas'
    print(f"--- {f0['ID']} | {f0['PRE_SEXO']}, {f0['PRE_EDAD']:.0f} años, {f0['PRE_ESTATURA']:.0f} cm, {senas_txt}")
    print(f"    Candidatos tras blocking: {len(P_demo)} de {len(bl):,} expedientes")
    cols = ['ID_CAND', 'dif_edad', 'dif_estatura', 'municipio_coincide', 'sim_senas', 'sim_ropa', 'score'] + \
           [c for c in P_demo.columns if c.startswith('score base')]
    display(P_demo.nlargest(5, 'score')[cols].rename(columns={'ID_CAND': 'Expediente (índice interno)'}).round(4))
    print(f"    Score máximo: {P_demo.score.max():.4g} -> "
          f"{'comparable con coincidencias verdaderas: prioridad alta de revisión.' if P_demo.score.max() >= ref_pos.quantile(0.10) else 'por debajo del percentil 10 de las coincidencias verdaderas: prioridad baja.'}\n")
print("El perito revisa estos candidatos y confirma o descarta con evidencia forense. El modelo no identifica por sí solo.")

## Conclusiones

La celda siguiente redacta las conclusiones con las cifras de esta ejecución. Las lecturas metodológicas que no
dependen de los números son estas:

- **El ensamble se justifica por evidencia, no por complejidad.** La rúbrica pide ensambles; el proceso pericial pide
  que cualquier complejidad adicional se pague con una mejora demostrable en el error que importa (falsos negativos).
  Por eso la mejora se mide con diferencia pareada y la elección conserva la regla de parsimonia.
- **Diversidad de errores.** Un ensamble heterogéneo solo puede superar a sus modelos base en las fichas donde estos
  se equivocan de forma distinta. El "Recall@10 oráculo" de 5.4 acota cuánto se podía ganar combinando los tres
  finalistas del Avance 4.
- **El límite sigue estando en la representación.** Si familias con sesgos tan distintos (lineales, árboles en
  *bagging* y en *boosting*, redes y sus combinaciones) convergen al mismo rango sobre las mismas 10 características,
  la siguiente mejora no viene del algoritmo sino de la información: embeddings semánticos de señas y ropa, y mejor
  captura de señas particulares en las necropsias.
- **Datos no vistos.** Todas las métricas finales se calcularon sobre las 107 pseudo-fichas de prueba, que no se
  usaron en ninguna búsqueda de hiperparámetros, y sobre tres variantes de ruido generadas aparte (5.6).

In [ ]:
# =============================================================================
# CONCLUSIONES AUTOMÁTICAS (con las cifras de esta ejecución)
# =============================================================================
f_fin, f_ind = tabla_5_5.loc[MODELO_FINAL_NOMBRE], tabla_5_5.loc[MEJOR_INDIVIDUAL]
mejor_h = max([m for m in ENSAMBLES if REGISTRO[m]['tipo'] == 'Ensamble homogéneo'], key=lambda m: tabla_5_5.loc[m, 'Recall@10'])
mejor_he = max([m for m in ENSAMBLES if REGISTRO[m]['tipo'] == 'Ensamble heterogéneo'], key=lambda m: tabla_5_5.loc[m, 'Recall@10'])
print(f"1. Se construyeron {len(ENSAMBLES)} ensambles: "
      f"{sum(REGISTRO[m]['tipo'] == 'Ensamble homogéneo' for m in ENSAMBLES)} homogéneos y "
      f"{sum(REGISTRO[m]['tipo'] == 'Ensamble heterogéneo' for m in ENSAMBLES)} heterogéneos, con optimización de "
      "hiperparámetros por validación cruzada agrupada por ficha. Los heterogéneos combinan los tres finalistas del Avance 4.")
print(f"2. Mejor homogéneo: {mejor_h} (Recall@10 = {tabla_5_5.loc[mejor_h, 'Recall@10']:.3f}); mejor heterogéneo: "
      f"{mejor_he} (Recall@10 = {tabla_5_5.loc[mejor_he, 'Recall@10']:.3f}); mejor individual: {MEJOR_INDIVIDUAL} "
      f"(Recall@10 = {f_ind['Recall@10']:.3f}).")
print(f"3. Mejoras estadísticamente significativas sobre el mejor individual: {signif if signif else 'ninguna'} "
      "(bootstrap pareado, B = 2,000, n = 107 fichas).")
print(f"4. Modelo final: {MODELO_FINAL_NOMBRE}. Recall@10 = {f_fin['Recall@10']:.3f} [{f_fin['IC95% inf.']:.3f}, "
      f"{f_fin['IC95% sup.']:.3f}], MRR = {f_fin['MRR']:.3f}, PR-AUC = {f_fin['PR-AUC (pares)']:.3f}, ruido alto = "
      f"{tabla_estres.loc[MODELO_FINAL_NOMBRE, 'alto']:.3f}; puntaje en la matriz de decisión = "
      f"{matriz.loc[MODELO_FINAL_NOMBRE, 'Puntaje ponderado']:.2f}.")
print(f"5. Frente al piso heurístico (filtrado manual aproximado): Recall@10 {f_fin['Recall@10']:.3f} vs. "
      f"{tabla_5_5.loc['Heurística TF-IDF (piso)', 'Recall@10']:.3f}; con ruido alto {tabla_estres.loc[MODELO_FINAL_NOMBRE, 'alto']:.3f} "
      f"vs. {tabla_estres.loc['Heurística TF-IDF (piso)', 'alto']:.3f}.")
print(f"6. Se cumple el umbral de negocio ({UMBRAL_NEGOCIO:.2f}) en el IC inferior: "
      f"{'sí' if f_fin['IC95% inf.'] >= UMBRAL_NEGOCIO else 'no'}. Margen hasta el techo del protocolo "
      f"({TECHO_PROTOCOLO:.3f}): {TECHO_PROTOCOLO - f_fin['Recall@10']:.3f} puntos.")
print(f"7. Residuos: {seg['Sin señas en la ficha'][R_FINAL > 10].mean():.0%} de los fallos del modelo final son fichas sin "
      "señas particulares. La prioridad para la siguiente fase es la información de entrada, no el algoritmo.")

## Declaración de uso de inteligencia artificial (resumen)

El registro completo está en `docs/declaracion_uso_ia.md` del repositorio, en formato Decisión / Justificación /
Alternativa descartada / Participación de la IA.

| Herramienta | Uso en este avance | Tipo de uso |
|---|---|---|
| Claude (Anthropic), en claude.ai | Diseño del protocolo de ensambles, redacción del código y de las interpretaciones automáticas, prueba del notebook con datos sintéticos | Asistente de trabajo |

**Decisión 28 — Ensambles del Avance 5 sobre el mismo vector de comparación.**
- *Decisión:* seis ensambles (Random Forest, Gradient Boosting, Bagging de regresión logística, votación suave,
  stacking y blending) sobre los mismos pares y las mismas 10 características del Avance 4; heterogéneos con los tres
  finalistas del Avance 4 en su configuración elegida por validación cruzada.
- *Justificación:* mantener los datos fijos aísla el efecto del ensamble; usar la configuración de validación cruzada
  evita seleccionar con el conjunto de prueba.
- *Alternativas descartadas:* usar el SVM y el MLP sin ajustar porque rindieron más en prueba (sería fuga de
  información); `StackingClassifier` con su `cv` por defecto (mezclaría pares de una misma ficha entre pliegues);
  LambdaMART/XGBRanker (agrega una dependencia fuera de scikit-learn; queda como trabajo futuro).
- *Participación de la IA:* propuso el protocolo y escribió el código; lo ejecutó de punta a punta solo con un archivo
  **sintético** con las mismas columnas para verificar que corre. **Ninguna cifra de este notebook proviene de la IA:**
  todas se generan al ejecutarlo el equipo con el archivo real, que no se compartió con la herramienta en este avance.

## Referencias (APA 7)

Breiman, L. (1996). Bagging predictors. *Machine Learning, 24*(2), 123–140. https://doi.org/10.1007/BF00058655

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Christen, P. (2012). *Data matching: Concepts and techniques for record linkage, entity resolution, and duplicate detection*. Springer. https://doi.org/10.1007/978-3-642-31164-2

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.

Fellegi, I. P., & Sunter, A. B. (1969). A theory for record linkage. *Journal of the American Statistical Association, 64*(328), 1183–1210. https://doi.org/10.1080/01621459.1969.10501049

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. In *Advances in Neural Information Processing Systems 30* (pp. 3146–3154). Curran Associates.

Niculescu-Mizil, A., & Caruana, R. (2005). Predicting good probabilities with supervised learning. In *Proceedings of the 22nd International Conference on Machine Learning* (pp. 625–632). https://doi.org/10.1145/1102351.1102430

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Platt, J. C. (1999). Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In A. J. Smola, P. Bartlett, B. Schölkopf, & D. Schuurmans (Eds.), *Advances in large margin classifiers* (pp. 61–74). MIT Press.

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

Studer, S., Bui, T. B., Drescher, C., Hanuschkin, A., Winkler, L., Peters, S., & Müller, K.-R. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction, 3*(2), 392–413. https://doi.org/10.3390/make3020020

Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241–259. https://doi.org/10.1016/S0893-6080(05)80023-1

Zhou, Z.-H. (2012). *Ensemble methods: Foundations and algorithms*. CRC Press.